# Day 12 - 1교시: 컬럼 조작과 필터링

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- `select()`로 필요한 컬럼만 선택할 수 있다
- `withColumn()`으로 새 컬럼을 추가하거나 수정할 수 있다
- `filter()`/`where()`로 데이터를 조건별로 필터링할 수 있다
- `when()`/`otherwise()`로 조건부 값을 설정할 수 있다
- 정렬, 중복 제거, 제한 등 기본 연산을 수행할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 생성 및 데이터 로드
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, upper, lower, concat, substring
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성 (이전 교시에서 이미 있으면 재사용)
spark = SparkSession.builder \
    .appName("PySpark-Column-Filter") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 생성 (이전 교시 데이터 없으면 새로 생성)
np.random.seed(42)

os.makedirs("/tmp/spark_tutorial", exist_ok=True)

# 샘플 데이터
sample_data = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 101)],
    "name": [f"Employee_{i}" for i in range(1, 101)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 100
    ),
    "salary": np.random.randint(40000, 120000, 100),
    "age": np.random.randint(25, 55, 100),
    "is_manager": np.random.choice([True, False], 100, p=[0.2, 0.8]),
})

# 결측치 추가
sample_data.loc[5:10, "salary"] = None
sample_data.loc[15:18, "department"] = None

sample_data.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"행 수: {df.count()}, 컬럼: {df.columns}")
df.show(5)

---

## Part 1: 컬럼 참조 방법

### col() 함수 vs 문자열 vs df.컬럼명

Spark에서 컬럼을 참조하는 방법은 3가지가 있습니다:

| 방법 | 예시 | 사용 상황 |
|------|------|----------|
| 문자열 | `"name"` | select, groupBy 등에서 단순 참조 |
| col() 함수 | `col("name")` | 연산, 조건식에서 사용 (권장) |
| df.컬럼명 | `df.name` | 조인 시 테이블 구분 |

In [ ]:
# -----------------------------------------------------------------------------
# 컬럼 참조 방법 비교
# -----------------------------------------------------------------------------
from pyspark.sql.functions import col

# 방법 1: 문자열 (가장 단순)
# select, groupBy 등에서 컬럼명만 필요할 때
df.select("name", "salary").show(3)

# 방법 2: col() 함수 (가장 권장)
# 연산이나 조건식에서 사용
# col("컬럼명"): Column 객체 반환 → 연산 가능
df.select(col("name"), col("salary") * 1.1).show(3)

# 방법 3: df.컬럼명 (조인 시 유용)
# 여러 DataFrame을 다룰 때 어떤 테이블의 컬럼인지 명확히
df.select(df.name, df.salary).show(3)

In [ ]:
# -----------------------------------------------------------------------------
# col() vs 문자열: 연산 가능 여부
# -----------------------------------------------------------------------------

# 문자열은 연산 불가 (에러 발생)
# df.select("salary" * 1.1)  # TypeError!

# col()은 연산 가능
# col("salary"): salary 컬럼을 Column 객체로 반환
# * 1.1: 모든 값에 1.1 곱하기
df.select(
    col("name"),                    # 이름 그대로
    col("salary"),                  # 원래 급여
    col("salary") * 1.1             # 급여 10% 인상
).show(5)

### 실습

#### 실습 1-1: 컬럼 참조 방법 비교

name과 age 컬럼을 col() 함수를 사용하여 선택하세요.

<details>
<summary>힌트</summary>

`df.select(col("컬럼1"), col("컬럼2"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.select(col("name"), col("age")).show(5)
```

</details>

#### 실습 1-2: 컬럼 연산

salary 컬럼에 2를 곱한 결과를 조회하세요.

<details>
<summary>힌트</summary>

`col("컬럼") * 숫자`로 연산 가능

</details>

<details>
<summary>모범 답안</summary>

```python
df.select(col("name"), col("salary"), col("salary") * 2).show(5)
```

</details>

#### 실습 1-3: df.컬럼명 사용

df.name과 df.department를 사용하여 두 컬럼을 조회하세요.

<details>
<summary>힌트</summary>

`df.select(df.컬럼1, df.컬럼2)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.select(df.name, df.department).show(5)
```

</details>

---

## Part 2: 컬럼 선택 (select)

### select() 사용법

```python
# 기본 형태
df.select("col1", "col2", "col3")
df.select(col("col1"), col("col2"))

# 모든 컬럼
df.select("*")

# 연산과 함께
df.select("name", (col("salary") * 1.1).alias("new_salary"))
```

In [ ]:
# -----------------------------------------------------------------------------
# select(): 기본 사용법
# -----------------------------------------------------------------------------

# 단일 컬럼 선택
# 결과: 해당 컬럼만 포함된 새 DataFrame 반환
df.select("name").show(5)

# 여러 컬럼 선택
# 쉼표로 구분하여 나열
df.select("emp_id", "name", "department").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 컬럼 연산과 별칭(alias)
# -----------------------------------------------------------------------------

# alias(): 컬럼에 새 이름 부여
# (col("salary") * 1.1).alias("raised_salary"): 계산 결과에 이름 지정
result = df.select(
    col("name"),                                    # 이름 그대로
    col("salary"),                                  # 원래 급여
    (col("salary") * 1.1).alias("raised_salary"),  # 10% 인상 급여 (새 이름)
    (col("salary") / 12).alias("monthly_salary")   # 월급 (새 이름)
)

print("=== 급여 계산 ===")
result.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 컬럼 순서 변경, 특정 컬럼 제외
# -----------------------------------------------------------------------------

# 컬럼 순서 변경: select에 원하는 순서로 나열
reordered = df.select("department", "name", "salary", "emp_id")
print("=== 컬럼 순서 변경 ===")
reordered.show(3)

# 특정 컬럼 제외: drop() 사용 (select의 반대)
# drop(): 지정한 컬럼을 제외한 나머지 반환
without_manager = df.drop("is_manager")
print("=== is_manager 컬럼 제외 ===")
without_manager.show(3)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 동적 컬럼 선택 (리스트 활용)
# -----------------------------------------------------------------------------

# 컬럼 목록을 변수로 관리
cols_to_select = ["emp_id", "name", "salary"]

# * 연산자로 리스트 언패킹
# *cols_to_select: ["a", "b"] → "a", "b"로 풀어줌
df.select(*cols_to_select).show(5)

# 조건에 따라 컬럼 선택
numeric_cols = ["salary", "age"]
df.select(*numeric_cols).describe().show()

### 실습

#### 실습 2-1: 기본 컬럼 선택

emp_id, name, salary 3개 컬럼만 선택하세요.

<details>
<summary>힌트</summary>

`df.select("컬럼1", "컬럼2", "컬럼3")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.select("emp_id", "name", "salary").show(5)
```

</details>

#### 실습 2-2: 컬럼 연산과 별칭

salary 컬럼을 1000으로 나눈 값을 "salary_k"라는 이름으로 조회하세요.

<details>
<summary>힌트</summary>

`(col("컬럼") / 값).alias("새이름")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.select(
    col("name"),
    col("salary"),
    (col("salary") / 1000).alias("salary_k")
).show(5)
```

</details>

#### 실습 2-3: 컬럼 제외

is_manager 컬럼을 제외한 나머지 컬럼을 조회하세요.

<details>
<summary>힌트</summary>

`df.drop("컬럼명")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.drop("is_manager").show(5)
```

</details>

#### 실습 2-4: 리스트로 컬럼 선택

컬럼 목록 리스트 `["name", "department"]`를 사용하여 해당 컬럼들을 선택하세요.

<details>
<summary>힌트</summary>

`*리스트`로 언패킹

</details>

<details>
<summary>모범 답안</summary>

```python
cols = ["name", "department"]
df.select(*cols).show(5)
```

</details>

---

## Part 3: 컬럼 추가/수정 (withColumn)

### Spark의 불변성 (Immutability)

```
**Spark DataFrame은 불변(Immutable)**

Pandas:
df["new_col"] = df["old_col"] * 2  # 원본 df 변경

Spark:
df.withColumn("new_col", col("old_col") * 2)  # 새 df 반환
df = df.withColumn(...)  # 결과를 다시 할당해야 유지

★ 중요: withColumn() 결과를 변수에 저장하지 않으면 사라짐!
```

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 새 컬럼 추가
# -----------------------------------------------------------------------------

# withColumn(컬럼명, 표현식): 새 컬럼 추가 또는 기존 컬럼 덮어쓰기
# - 첫 번째 인자: 새 컬럼 이름 (문자열)
# - 두 번째 인자: 컬럼 값을 계산하는 표현식 (Column 객체)

# 새 컬럼 추가: 연봉을 월급으로 변환
# col("salary") / 12: salary 컬럼의 모든 값을 12로 나눔
df_with_monthly = df.withColumn(
    "monthly_salary",       # 새 컬럼 이름
    col("salary") / 12      # 계산식
)

print("=== 월급 컬럼 추가 ===")
df_with_monthly.select("name", "salary", "monthly_salary").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 상수값 추가 (lit 함수)
# -----------------------------------------------------------------------------

# lit(): 상수(리터럴) 값을 Column으로 변환
# - col("name") → 컬럼 참조
# - lit("Korea") → 상수값 "Korea"
#
# 모든 행에 같은 값을 넣을 때 사용

# 상수 컬럼 추가
df_with_country = df.withColumn(
    "country",      # 컬럼명
    lit("Korea")    # 모든 행에 "Korea" 값
)

print("=== 상수 컬럼 추가 ===")
df_with_country.select("name", "country").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 기존 컬럼 수정 (덮어쓰기)
# -----------------------------------------------------------------------------

# 같은 컬럼명을 사용하면 기존 컬럼을 덮어씀
# salary 컬럼을 10% 인상된 값으로 교체
df_raised = df.withColumn(
    "salary",           # 기존 컬럼명 (덮어쓰기)
    col("salary") * 1.1 # 10% 인상
)

print("=== 급여 10% 인상 (원본 비교) ===")
print("원본:")
df.select("name", "salary").show(3)
print("수정 후:")
df_raised.select("name", "salary").show(3)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 체이닝 (여러 컬럼 한번에)
# -----------------------------------------------------------------------------

# 여러 withColumn을 연속으로 호출 (메서드 체이닝)
# 각 withColumn은 새 DataFrame을 반환하므로 연결 가능
df_enhanced = (
    df
    # 연봉에서 월급 계산
    .withColumn("monthly_salary", col("salary") / 12)
    # 연봉에서 일급 계산 (연 250일 근무 가정)
    .withColumn("daily_salary", col("salary") / 250)
    # 국가 추가
    .withColumn("country", lit("Korea"))
    # 연도 추가
    .withColumn("year", lit(2024))
)

print("=== 여러 컬럼 추가 ===")
df_enhanced.show(5)

### 실습

#### 실습 3-1: 새 컬럼 추가

age에 1을 더한 "next_age" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`df.withColumn("새컬럼명", col("기존컬럼") + 값)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("next_age", col("age") + 1).select("name", "age", "next_age").show(5)
```

</details>

#### 실습 3-2: 상수 컬럼 추가



모든 행에 "2024"라는 값을 가진 "year" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`lit()` 함수로 상수값을 컬럼으로 변환

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("year", lit("2024")).select("name", "year").show(5)
```

</details>

#### 실습 3-3: 기존 컬럼 수정

age 컬럼의 값을 모두 10 증가시킨 DataFrame을 만드세요.

<details>
<summary>힌트</summary>

같은 컬럼명으로 withColumn하면 덮어쓰기

</details>

<details>
<summary>모범 답안</summary>

```python
df_aged = df.withColumn("age", col("age") + 10)
df_aged.select("name", "age").show(5)
```

</details>

#### 실습 3-4: 여러 컬럼 추가 (체이닝)

salary의 20%를 "tax", salary에서 tax를 뺀 "net_salary" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

withColumn을 체이닝하여 연속 호출

</details>

<details>
<summary>모범 답안</summary>

```python
df_tax = (
    df
    .withColumn("tax", col("salary") * 0.2)
    .withColumn("net_salary", col("salary") - col("salary") * 0.2)
)
df_tax.select("name", "salary", "tax", "net_salary").show(5)
```

</details>

---

## Part 4: 컬럼 이름 변경과 삭제

| 메서드 | 용도 | 예시 |
|--------|------|------|
| `withColumnRenamed()` | 단일 컬럼 이름 변경 | `df.withColumnRenamed("old", "new")` |
| `toDF()` | 모든 컬럼 이름 변경 | `df.toDF("a", "b", "c")` |
| `drop()` | 컬럼 삭제 | `df.drop("col1", "col2")` |

In [ ]:
# -----------------------------------------------------------------------------
# withColumnRenamed(): 컬럼 이름 변경
# -----------------------------------------------------------------------------

# withColumnRenamed(기존이름, 새이름): 단일 컬럼 이름 변경
# 원본 DataFrame은 변경되지 않음 (새 DataFrame 반환)
df_renamed = df.withColumnRenamed("emp_id", "employee_id")

print("=== 컬럼명 변경: emp_id → employee_id ===")
df_renamed.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# 여러 컬럼 이름 변경
# -----------------------------------------------------------------------------

# 방법 1: withColumnRenamed 체이닝
df_multi_renamed = (
    df
    .withColumnRenamed("emp_id", "employee_id")
    .withColumnRenamed("department", "dept")
    .withColumnRenamed("is_manager", "manager_flag")
)

print("=== 여러 컬럼명 변경 ===")
df_multi_renamed.printSchema()

# 방법 2: toDF() - 모든 컬럼명 한번에 변경
# 주의: 컬럼 순서와 개수가 정확히 일치해야 함
# df.toDF("new_col1", "new_col2", ...) - 모든 컬럼에 새 이름 지정

In [ ]:
# -----------------------------------------------------------------------------
# drop(): 컬럼 삭제
# -----------------------------------------------------------------------------

# drop(컬럼명): 지정한 컬럼 제거
# 여러 컬럼 제거: drop("col1", "col2") 또는 drop("col1").drop("col2")

# 단일 컬럼 삭제
df_no_manager = df.drop("is_manager")

print("=== is_manager 컬럼 삭제 ===")
print(f"삭제 전 컬럼: {df.columns}")
print(f"삭제 후 컬럼: {df_no_manager.columns}")

# 여러 컬럼 삭제
df_minimal = df.drop("is_manager", "age")
print(f"여러 컬럼 삭제 후: {df_minimal.columns}")

### 실습

#### 실습 4-1: 컬럼 이름 변경

salary 컬럼의 이름을 "annual_salary"로 변경하세요.

<details>
<summary>힌트</summary>

`df.withColumnRenamed("기존이름", "새이름")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumnRenamed("salary", "annual_salary").printSchema()
```

</details>

#### 실습 4-2: 여러 컬럼 이름 변경

emp_id를 "id"로, name을 "employee_name"으로 변경하세요.

<details>
<summary>힌트</summary>

withColumnRenamed를 체이닝

</details>

<details>
<summary>모범 답안</summary>

```python
df_renamed = (
    df
    .withColumnRenamed("emp_id", "id")
    .withColumnRenamed("name", "employee_name")
)
df_renamed.printSchema()
```

</details>

#### 실습 4-3: 컬럼 삭제

age와 is_manager 두 컬럼을 삭제하세요.

<details>
<summary>힌트</summary>

`df.drop("컬럼1", "컬럼2")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.drop("age", "is_manager").columns
```

</details>

---

## Part 5: 필터링 (filter / where)

### filter()와 where()는 동일

```python
df.filter(조건)   # 함수형 스타일
df.where(조건)    # SQL 스타일 (동일한 기능)
```

### 비교 연산자

| 연산자 | 의미 | 예시 |
|--------|------|------|
| `==` | 같음 | `col("dept") == "Sales"` |
| `!=` | 다름 | `col("dept") != "HR"` |
| `>`, `>=` | 크다, 크거나 같다 | `col("age") >= 30` |
| `<`, `<=` | 작다, 작거나 같다 | `col("salary") < 50000` |

### 논리 연산자

| 연산자 | 의미 | 주의사항 |
|--------|------|----------|
| `&` | AND | 괄호 필수! |
| `\|` | OR | 괄호 필수! |
| `~` | NOT | |

In [ ]:
# -----------------------------------------------------------------------------
# filter(): 기본 필터링
# -----------------------------------------------------------------------------

# 조건: col("컬럼") 연산자 값
# 결과: 조건을 만족하는 행만 포함된 새 DataFrame

# 급여가 80000 이상인 직원
high_salary = df.filter(col("salary") >= 80000)
print(f"=== 고연봉자 (80000 이상): {high_salary.count()}명 ===")
high_salary.show(5)

# 특정 부서 직원
engineers = df.filter(col("department") == "Engineering")
print(f"=== Engineering 부서: {engineers.count()}명 ===")
engineers.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# filter(): AND / OR 조건 (괄호 필수!)
# -----------------------------------------------------------------------------

# AND 조건: & 연산자 (각 조건을 괄호로 감싸야 함!)
# 이유: Python 연산자 우선순위 때문
# 틀린 예: col("age") > 30 & col("salary") > 70000  # 에러!
# 맞는 예: (col("age") > 30) & (col("salary") > 70000)

# AND: 30세 이상이면서 급여 70000 이상
senior_high = df.filter(
    (col("age") >= 30) & (col("salary") >= 70000)
)
print(f"=== 30세 이상 AND 고연봉: {senior_high.count()}명 ===")
senior_high.show(5)

# OR: Engineering이거나 Sales 부서
eng_or_sales = df.filter(
    (col("department") == "Engineering") | (col("department") == "Sales")
)
print(f"=== Engineering OR Sales: {eng_or_sales.count()}명 ===")

In [ ]:
# -----------------------------------------------------------------------------
# filter(): SQL 스타일 문자열 조건
# -----------------------------------------------------------------------------

# 문자열로 SQL WHERE 절처럼 작성 가능
# 더 직관적일 수 있음 (SQL에 익숙하면)

# SQL 스타일 필터링
df.filter("age >= 30 AND salary >= 70000").show(5)

# SQL 스타일: BETWEEN
df.filter("salary BETWEEN 50000 AND 80000").show(5)

# SQL 스타일: IN
df.filter("department IN ('Engineering', 'Sales', 'Marketing')").show(5)

# SQL 스타일: LIKE (문자열 패턴)
df.filter("name LIKE 'Employee_1%'").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# filter(): isin(), isNull(), isNotNull()
# -----------------------------------------------------------------------------

# isin(): 여러 값 중 하나인지 확인
# SQL의 IN 절과 동일
target_depts = ["Engineering", "Sales"]
df.filter(col("department").isin(target_depts)).show(5)

# isin()에 리스트 직접 전달
df.filter(col("department").isin("HR", "Finance")).show(5)

# isNull(): NULL인 행만
df.filter(col("salary").isNull()).show()

# isNotNull(): NULL이 아닌 행만
df.filter(col("salary").isNotNull()).count()

In [ ]:
# -----------------------------------------------------------------------------
# filter(): 문자열 조건 메서드
# -----------------------------------------------------------------------------

# startswith(): 특정 문자로 시작
df.filter(col("name").startswith("Employee_1")).show(5)

# endswith(): 특정 문자로 끝
df.filter(col("emp_id").endswith("5")).show(5)

# contains(): 특정 문자 포함
df.filter(col("department").contains("ing")).show(5)  # Engineering, Marketing

In [ ]:
# -----------------------------------------------------------------------------
# where(): filter()와 동일 (SQL 친화적 이름)
# -----------------------------------------------------------------------------

# where()는 filter()의 별칭 (alias)
# SQL에 익숙한 사람을 위한 이름

# filter()와 완전히 동일한 동작
df.where(col("age") >= 30).show(5)
df.where("salary > 60000").show(5)

### 실습

#### 실습 5-1: 기본 필터링

salary가 70000 이상인 직원만 조회하세요.

<details>
<summary>힌트</summary>

`df.filter(col("컬럼") >= 값)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.filter(col("salary") >= 70000).select("name", "salary").show()
```

</details>

#### 실습 5-2: AND 조건

department가 "Engineering"이면서 age가 35 이상인 직원을 조회하세요.

<details>
<summary>힌트</summary>

`(조건1) & (조건2)` - 괄호 필수!

</details>

<details>
<summary>모범 답안</summary>

```python
df.filter(
    (col("department") == "Engineering") & (col("age") >= 35)
).select("name", "department", "age").show()
```

</details>

#### 실습 5-3: OR 조건

department가 "Sales" 또는 "Marketing"인 직원을 조회하세요.

<details>
<summary>힌트</summary>

`(조건1) | (조건2)` 또는 `isin()` 사용

</details>

<details>
<summary>모범 답안</summary>

```python
# 방법 1: OR 조건
df.filter(
    (col("department") == "Sales") | (col("department") == "Marketing")
).show()

# 방법 2: isin
df.filter(col("department").isin("Sales", "Marketing")).show()
```

</details>

#### 실습 5-4: NULL 필터링

salary가 NULL인 행을 조회하세요.

<details>
<summary>힌트</summary>

`col("컬럼").isNull()`

</details>

<details>
<summary>모범 답안</summary>

```python
df.filter(col("salary").isNull()).select("emp_id", "name", "salary").show()
```

</details>

#### 실습 5-5: 문자열 조건

name이 "Employee_1"로 시작하는 직원을 조회하세요.

<details>
<summary>힌트</summary>

`col("컬럼").startswith("문자열")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.filter(col("name").startswith("Employee_1")).select("name").show()
```

</details>

---

## Part 6: 조건부 값 설정 (when / otherwise)

### when/otherwise = SQL의 CASE WHEN

```
SQL:
CASE
    WHEN age >= 50 THEN 'Senior'
    WHEN age >= 30 THEN 'Middle'
    ELSE 'Junior'
END

PySpark:
when(col("age") >= 50, "Senior")
.when(col("age") >= 30, "Middle")
.otherwise("Junior")
```

In [ ]:
# -----------------------------------------------------------------------------
# when(): 단일 조건
# -----------------------------------------------------------------------------

# when(조건, 참일때값): 조건이 참이면 지정값, 거짓이면 null
# otherwise(값): 모든 when 조건이 거짓일 때 값

# 성인 여부 판단
df_adult = df.withColumn(
    "is_adult",
    # when(조건, 참일때값).otherwise(거짓일때값)
    when(col("age") >= 18, "Yes").otherwise("No")
)

df_adult.select("name", "age", "is_adult").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# when(): 다중 조건 (if-elif-else)
# -----------------------------------------------------------------------------

# 여러 when을 체이닝: 첫 번째로 참인 조건의 값 반환
# otherwise: 모든 조건이 거짓일 때 값 (생략 시 null)

# 연령대 분류
df_age_group = df.withColumn(
    "age_group",
    # 첫 번째 참인 조건에서 멈춤
    when(col("age") >= 50, "50대 이상")      # 50 이상이면 "50대 이상"
    .when(col("age") >= 40, "40대")          # 40~49면 "40대"
    .when(col("age") >= 30, "30대")          # 30~39면 "30대"
    .otherwise("20대")                        # 나머지는 "20대"
)

print("=== 연령대 분류 ===")
df_age_group.select("name", "age", "age_group").show(10)

# 연령대별 인원 수
df_age_group.groupBy("age_group").count().show()

In [ ]:
# -----------------------------------------------------------------------------
# when(): 급여 등급 분류 (실무 예제)
# -----------------------------------------------------------------------------

# 급여 등급 분류
df_salary_grade = df.withColumn(
    "salary_grade",
    when(col("salary") >= 100000, "S")       # 10만 이상: S등급
    .when(col("salary") >= 80000, "A")       # 8만~10만: A등급
    .when(col("salary") >= 60000, "B")       # 6만~8만: B등급
    .when(col("salary") >= 40000, "C")       # 4만~6만: C등급
    .otherwise("D")                           # 4만 미만: D등급
)

print("=== 급여 등급 ===")
df_salary_grade.select("name", "salary", "salary_grade").show(10)

# 등급별 인원 분포
df_salary_grade.groupBy("salary_grade").count().orderBy("salary_grade").show()

In [ ]:
# -----------------------------------------------------------------------------
# when(): NULL 처리와 결합
# -----------------------------------------------------------------------------

# NULL을 특정 값으로 대체하면서 조건 분기
df_null_handled = df.withColumn(
    "salary_status",
    when(col("salary").isNull(), "미입력")           # NULL이면 "미입력"
    .when(col("salary") >= 80000, "고연봉")          # 8만 이상
    .when(col("salary") >= 50000, "중연봉")          # 5만~8만
    .otherwise("저연봉")                              # 5만 미만
)

print("=== NULL 처리 포함 급여 상태 ===")
df_null_handled.select("name", "salary", "salary_status").show(15)

### 실습

#### 실습 6-1: 단일 조건

is_manager가 True이면 "Manager", 아니면 "Staff"인 "role" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`when(조건, 참값).otherwise(거짓값)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn(
    "role",
    when(col("is_manager") == True, "Manager").otherwise("Staff")
).select("name", "is_manager", "role").show(5)
```

</details>

#### 실습 6-2: 다중 조건

age 기준으로 "age_band" 컬럼을 추가하세요:
- 40 이상: "40+"
- 30 이상: "30s"
- 그 외: "20s"

<details>
<summary>힌트</summary>

`when().when().otherwise()` 체이닝

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn(
    "age_band",
    when(col("age") >= 40, "40+")
    .when(col("age") >= 30, "30s")
    .otherwise("20s")
).select("name", "age", "age_band").show(10)
```

</details>

#### 실습 6-3: 급여 구간 분류

salary 기준으로 "salary_level" 컬럼을 추가하세요:
- 90000 이상: "High"
- 60000 이상: "Medium"
- 그 외: "Low"

<details>
<summary>힌트</summary>

when().when().otherwise() 체이닝

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn(
    "salary_level",
    when(col("salary") >= 90000, "High")
    .when(col("salary") >= 60000, "Medium")
    .otherwise("Low")
).select("name", "salary", "salary_level").show(10)
```

</details>

---

## Part 7: 정렬, 중복 제거, 제한

| 메서드 | 용도 | 예시 |
|--------|------|------|
| `orderBy()` / `sort()` | 정렬 | `df.orderBy(col("salary").desc())` |
| `distinct()` | 전체 행 중복 제거 | `df.distinct()` |
| `dropDuplicates()` | 특정 컬럼 기준 중복 제거 | `df.dropDuplicates(["col"])` |
| `limit()` | 상위 N개 | `df.limit(10)` |

In [ ]:
# -----------------------------------------------------------------------------
# orderBy(): 정렬
# -----------------------------------------------------------------------------

# orderBy(): 지정 컬럼 기준 정렬
# - 기본: 오름차순 (ASC)
# - 내림차순: col("컬럼").desc()

# 급여 기준 오름차순 (기본)
df.orderBy("salary").select("name", "salary").show(5)

# 급여 기준 내림차순
df.orderBy(col("salary").desc()).select("name", "salary").show(5)

# 여러 컬럼 정렬: 부서 오름차순 → 급여 내림차순
df.orderBy(
    col("department").asc(),    # 부서 오름차순
    col("salary").desc()        # 급여 내림차순
).select("department", "name", "salary").show(10)

In [ ]:
# -----------------------------------------------------------------------------
# distinct(): 중복 제거
# -----------------------------------------------------------------------------

# distinct(): 전체 행이 동일한 중복 제거
# 반환: 고유한 행만 포함된 DataFrame

# 부서 목록 (고유값)
departments = df.select("department").distinct()
print("=== 부서 목록 ===")
departments.show()

# dropDuplicates(): 특정 컬럼 기준 중복 제거
# 해당 컬럼 값이 같은 행 중 첫 번째만 유지
df.dropDuplicates(["department"]).select("department", "name").show()

In [ ]:
# -----------------------------------------------------------------------------
# limit(): 상위 N개
# -----------------------------------------------------------------------------

# limit(n): 상위 n개 행만 반환
# 주의: 정렬 없이 limit만 쓰면 순서가 보장되지 않음

# 상위 5개
df.limit(5).show()

# 급여 상위 5명 (정렬 후 limit)
df.orderBy(col("salary").desc()).limit(5).select("name", "salary").show()

In [ ]:
# -----------------------------------------------------------------------------
# 실무 패턴: Top N 뽑기
# -----------------------------------------------------------------------------

# 부서별 최고 연봉자 1명씩 (간단 버전)
# 실제로는 Window 함수 사용이 더 정확함 (4교시에서 다룸)

# 정렬 후 부서별 첫 번째 행만
top_by_dept = (
    df
    .orderBy(col("salary").desc())
    .dropDuplicates(["department"])
    .select("department", "name", "salary")
    .orderBy("department")
)

print("=== 부서별 최고 연봉자 (간단 버전) ===")
top_by_dept.show()

### 실습

#### 실습 7-1: 정렬

salary 기준 내림차순으로 정렬하여 상위 10명을 조회하세요.

<details>
<summary>힌트</summary>

`df.orderBy(col("컬럼").desc()).limit(N)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.orderBy(col("salary").desc()).limit(10).select("name", "salary").show()
```

</details>

#### 실습 7-2: 다중 정렬

department 오름차순, age 내림차순으로 정렬하세요.

<details>
<summary>힌트</summary>

`orderBy(col("컬럼1").asc(), col("컬럼2").desc())`

</details>

<details>
<summary>모범 답안</summary>

```python
df.orderBy(
    col("department").asc(),
    col("age").desc()
).select("department", "name", "age").show(10)
```

</details>

#### 실습 7-3: 중복 제거

고유한 department 목록을 조회하세요.

<details>
<summary>힌트</summary>

`df.select("컬럼").distinct()`

</details>

<details>
<summary>모범 답안</summary>

```python
df.select("department").distinct().show()
```

</details>

#### 실습 7-4: 연봉 상위 3명

급여가 가장 높은 직원 3명의 이름과 급여를 조회하세요.

<details>
<summary>힌트</summary>

정렬 후 limit() 사용

</details>

<details>
<summary>모범 답안</summary>

```python
df.orderBy(col("salary").desc()).limit(3).select("name", "salary").show()
```

</details>

---

## 퀴즈

Q1. PySpark에서 새 컬럼을 추가할 때 올바른 방법은?

- A) `df["new_col"] = df["old_col"] * 2`
- B) `df.withColumn("new_col", col("old_col") * 2)`
- C) `df.addColumn("new_col", "old_col * 2")`
- D) `df.new_col = df.old_col * 2`

<details>
<summary>정답 보기</summary>

**정답: B) `df.withColumn("new_col", col("old_col") * 2)`**

PySpark DataFrame은 불변(immutable)이라 Pandas처럼 직접 할당할 수 없습니다.
`withColumn()`으로 새 DataFrame을 생성해야 합니다.

</details>

---

Q2. 다음 코드의 문제점은?

```python
df.filter(col("age") > 30 & col("salary") > 50000)
```

- A) filter() 대신 where()를 써야 한다
- B) & 연산자 양쪽에 괄호가 필요하다
- C) col() 대신 문자열을 써야 한다
- D) 문제없이 잘 동작한다

<details>
<summary>정답 보기</summary>

**정답: B) & 연산자 양쪽에 괄호가 필요하다**

Python 연산자 우선순위 때문에 `col("age") > 30 & col("salary")`처럼 해석됩니다.
올바른 코드: `(col("age") > 30) & (col("salary") > 50000)`

</details>

---

Q3. 모든 행에 상수값 "Korea"를 추가하려면?

- A) `df.withColumn("country", "Korea")`
- B) `df.withColumn("country", col("Korea"))`
- C) `df.withColumn("country", lit("Korea"))`
- D) `df.addColumn("country", "Korea")`

<details>
<summary>정답 보기</summary>

**정답: C) `df.withColumn("country", lit("Korea"))`**

`lit()` 함수는 상수(리터럴) 값을 Column 객체로 변환합니다.
문자열 그대로 넣으면 컬럼명으로 해석되어 에러가 발생합니다.

</details>

---

Q4. when/otherwise 표현식에서 모든 조건이 거짓일 때 otherwise()를 생략하면?

- A) 에러가 발생한다
- B) 빈 문자열("")이 된다
- C) 0이 된다
- D) null이 된다

<details>
<summary>정답 보기</summary>

**정답: D) null이 된다**

otherwise()를 생략하면 모든 when 조건이 거짓일 때 null이 반환됩니다.
명시적으로 처리하려면 otherwise()를 항상 작성하는 것이 좋습니다.

</details>

---

## 과제: 인사팀 월간 리포트 생성

### 시나리오

HR팀에서 경영진에게 보고할 **월간 인사 현황 리포트**를 요청했습니다.
원본 직원 데이터를 정제하고, 분석에 필요한 파생 컬럼을 추가한 뒤,
특정 조건의 직원만 필터링하여 최종 리포트 DataFrame을 생성하세요.

**요구사항:**
- 리포트에 필요한 컬럼만 선택
- 컬럼명을 비즈니스 친화적으로 변경
- 급여 등급 및 세후 급여 계산
- 고성과자(특정 조건) 필터링
- 급여 순으로 정렬

---

### Step 1: 컬럼 선택 및 이름 정리

리포트에 필요한 컬럼(`emp_id`, `name`, `department`, `salary`, `age`)만 선택하고,
비즈니스 친화적인 이름으로 변경하세요:
- `emp_id` → `사원번호`
- `name` → `이름`
- `department` → `부서`
- `salary` → `연봉`
- `age` → `나이`

<details>
<summary>힌트</summary>

`select()` 후 `withColumnRenamed()`를 체이닝하거나, `alias()`를 사용

</details>

<details>
<summary>모범 답안</summary>

```python
df_step1 = (
    df.select(
        col("emp_id").alias("사원번호"),
        col("name").alias("이름"),
        col("department").alias("부서"),
        col("salary").alias("연봉"),
        col("age").alias("나이")
    )
)
df_step1.show(5)
```

</details>

---

### Step 2: 파생 컬럼 추가

다음 파생 컬럼을 추가하세요:

1. **급여등급**: 연봉 기준으로 분류
   - 100,000 이상: "S"
   - 80,000 이상: "A"
   - 60,000 이상: "B"
   - 그 외: "C"

2. **세후연봉**: 연봉의 80% (세금 20% 가정)

3. **월급**: 연봉 / 12 (소수점 없이 정수로)

<details>
<summary>힌트</summary>

- `when().when().when().otherwise()` 체이닝
- `withColumn()`으로 계산식 추가
- `cast("int")` 또는 `round()`로 정수 변환

</details>

<details>
<summary>모범 답안</summary>

```python
df_step2 = (
    df_step1
    .withColumn(
        "급여등급",
        when(col("연봉") >= 100000, "S")
        .when(col("연봉") >= 80000, "A")
        .when(col("연봉") >= 60000, "B")
        .otherwise("C")
    )
    .withColumn("세후연봉", (col("연봉") * 0.8).cast("int"))
    .withColumn("월급", (col("연봉") / 12).cast("int"))
)
df_step2.show(5)
```

</details>

---

### Step 3: 고성과자 필터링

다음 조건을 **모두** 만족하는 직원만 필터링하세요:
- 급여등급이 "S" 또는 "A"
- 나이가 35세 이상
- 부서가 NULL이 아닌 경우

<details>
<summary>힌트</summary>

- `isin()`으로 여러 값 비교
- `&` 연산자로 AND 조건 결합 (괄호 필수!)
- `isNotNull()`로 NULL 체크

</details>

<details>
<summary>모범 답안</summary>

```python
df_step3 = df_step2.filter(
    (col("급여등급").isin("S", "A")) &
    (col("나이") >= 35) &
    (col("부서").isNotNull())
)
df_step3.show()
print(f"고성과자 수: {df_step3.count()}명")
```

</details>

---

### Step 4: 정렬 및 최종 정리

최종 리포트를 다음과 같이 정리하세요:
1. 연봉 내림차순, 같은 연봉이면 이름 오름차순으로 정렬
2. 상위 20명만 추출
3. `나이` 컬럼 제외 (리포트에서 제외 요청)

<details>
<summary>힌트</summary>

- `orderBy(col().desc(), col().asc())`
- `limit()`
- `drop()`

</details>

<details>
<summary>모범 답안</summary>

```python
df_step4 = (
    df_step3
    .orderBy(col("연봉").desc(), col("이름").asc())
    .limit(20)
    .drop("나이")
)
df_step4.show()
```

</details>

---

### 최종 과제: 전체 파이프라인 완성

위 Step 1~4를 하나의 체이닝된 코드로 작성하여 최종 리포트를 생성하세요.

<details>
<summary>힌트</summary>

모든 transformation을 괄호 안에서 `.`으로 연결

</details>

<details>
<summary>모범 답안</summary>

```python
# 전체 파이프라인
df_report = (
    df
    # Step 1: 컬럼 선택 및 이름 정리
    .select(
        col("emp_id").alias("사원번호"),
        col("name").alias("이름"),
        col("department").alias("부서"),
        col("salary").alias("연봉"),
        col("age").alias("나이")
    )
    # Step 2: 파생 컬럼 추가
    .withColumn(
        "급여등급",
        when(col("연봉") >= 100000, "S")
        .when(col("연봉") >= 80000, "A")
        .when(col("연봉") >= 60000, "B")
        .otherwise("C")
    )
    .withColumn("세후연봉", (col("연봉") * 0.8).cast("int"))
    .withColumn("월급", (col("연봉") / 12).cast("int"))
    # Step 3: 필터링
    .filter(
        (col("급여등급").isin("S", "A")) &
        (col("나이") >= 35) &
        (col("부서").isNotNull())
    )
    # Step 4: 정렬 및 정리
    .orderBy(col("연봉").desc(), col("이름").asc())
    .limit(20)
    .drop("나이")
)

print("=== 최종 인사 리포트 ===")
df_report.show()
print(f"리포트 대상 인원: {df_report.count()}명")
```

</details>

---

## 핵심 요약

### 컬럼 참조

```python
col("컬럼명")  # 가장 권장 (연산 가능)
"컬럼명"      # select, groupBy에서 단순 참조
df.컬럼명     # 조인 시 테이블 구분
```

### 컬럼 조작

| 작업 | 코드 |
|------|------|
| 선택 | `df.select("col1", "col2")` |
| 추가 | `df.withColumn("new", col("old") * 2)` |
| 상수 추가 | `df.withColumn("country", lit("Korea"))` |
| 이름 변경 | `df.withColumnRenamed("old", "new")` |
| 삭제 | `df.drop("col1", "col2")` |

### 필터링

```python
df.filter(col("age") >= 30)                          # 기본
df.filter((col("age") >= 30) & (col("salary") > 50000))  # AND (괄호 필수!)
df.filter(col("dept").isin("A", "B"))                # IN
df.filter("age >= 30 AND salary > 50000")            # SQL 스타일
```

### 조건부 값 (when/otherwise)

```python
when(col("age") >= 50, "Senior")
.when(col("age") >= 30, "Middle")
.otherwise("Junior")
```

### 정렬/중복/제한

```python
df.orderBy(col("salary").desc())     # 정렬
df.distinct()                         # 중복 제거
df.dropDuplicates(["col"])           # 특정 컬럼 기준 중복 제거
df.limit(10)                         # 상위 10개
```

In [ ]:
# 세션 유지 (다음 교시 계속)
print("\n2교시 완료! 다음 교시: 집계와 조인")

# Day 12 - 2교시: 집계와 조인

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- `groupBy()`와 `agg()`로 그룹별 집계를 수행할 수 있다
- `count()`, `sum()`, `avg()`, `min()`, `max()` 집계 함수를 활용할 수 있다
- 다양한 조인 타입(inner, left, right, outer)을 이해하고 적용할 수 있다
- `pivot()`으로 데이터를 피벗 테이블 형태로 변환할 수 있다
- Spark SQL을 DataFrame과 함께 활용할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 및 테스트 데이터 준비
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, count, sum, avg, min, max,
    countDistinct, first, last, collect_list, collect_set,
    round as spark_round, expr
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Aggregation-Join") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 디렉토리
os.makedirs("/tmp/spark_tutorial", exist_ok=True)

np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터 1: 직원 정보
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 51)],
    "name": [f"Employee_{i}" for i in range(1, 51)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 50
    ),
    "salary": np.random.randint(40000, 120000, 50),
    "age": np.random.randint(25, 55, 50),
    "hire_year": np.random.choice([2020, 2021, 2022, 2023, 2024], 50),
})
employees.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 2: 부서 정보 (조인용)
# -----------------------------------------------------------------------------
departments = pd.DataFrame({
    "dept_name": ["Engineering", "Sales", "Marketing", "HR", "Finance", "Legal"],
    "dept_head": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "budget": [500000, 300000, 200000, 150000, 400000, 100000],
    "location": ["Seoul", "Busan", "Seoul", "Daegu", "Seoul", "Incheon"],
})
departments.to_csv("/tmp/spark_tutorial/departments.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 3: 매출 데이터 (시계열)
# -----------------------------------------------------------------------------
sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=100, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 100),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 100),
    "amount": np.random.randint(100, 1000, 100),
    "quantity": np.random.randint(1, 20, 100),
})
sales.to_csv("/tmp/spark_tutorial/sales.csv", index=False)

# DataFrame 로드
df_emp = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)
df_dept = spark.read.csv("/tmp/spark_tutorial/departments.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df_emp.count()}명, 부서: {df_dept.count()}개, 매출: {df_sales.count()}건")

---

## Part 1: 전체 집계 (agg)

### 집계 함수 종류

| 함수 | 설명 | 예시 |
|------|------|------|
| `count(col)` | 개수 (NULL 제외) | `count("*")` 또는 `count("col")` |
| `sum(col)` | 합계 | `sum("salary")` |
| `avg(col)` | 평균 | `avg("age")` |
| `min(col)` | 최소값 | `min("salary")` |
| `max(col)` | 최대값 | `max("salary")` |
| `countDistinct(col)` | 고유값 개수 | `countDistinct("department")` |

In [ ]:
# -----------------------------------------------------------------------------
# agg(): 전체 데이터 집계 (groupBy 없이)
# -----------------------------------------------------------------------------

# agg(): 집계 함수들을 적용하여 결과 DataFrame 반환
# 여러 집계를 한 번에 계산 가능

# 전체 직원 통계
total_stats = df_emp.agg(
    count("*").alias("총인원"),                    # 전체 행 수
    count("salary").alias("급여있는인원"),          # NULL 제외 카운트
    sum("salary").alias("총급여"),                 # 급여 합계
    avg("salary").alias("평균급여"),               # 급여 평균
    min("salary").alias("최소급여"),               # 최소 급여
    max("salary").alias("최대급여"),               # 최대 급여
    countDistinct("department").alias("부서수"),   # 고유 부서 수
)

print("=== 전체 직원 통계 ===")
total_stats.show()

In [ ]:
# -----------------------------------------------------------------------------
# 단일 집계 함수 사용 (shortcut)
# -----------------------------------------------------------------------------

# 간단한 집계는 agg() 없이 직접 호출 가능
# 결과는 단일 값이 아닌 DataFrame

# 전체 행 수
print(f"전체 직원 수: {df_emp.count()}")

# 고유 부서 수
unique_depts = df_emp.select("department").distinct().count()
print(f"부서 종류: {unique_depts}개")

### 실습

#### 실습 1-1: 기본 집계

직원 데이터의 총 행 수와 평균 급여를 한 번에 조회하세요.

<details>
<summary>힌트</summary>

`df.agg(count("*"), avg("컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.agg(
    count("*").alias("총인원"),
    avg("salary").alias("평균급여")
).show()
```

</details>

#### 실습 1-2: 여러 집계 함수

salary의 최소값, 최대값, 합계를 한 번에 조회하세요.

<details>
<summary>힌트</summary>

`min()`, `max()`, `sum()` 함수 사용

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.agg(
    min("salary").alias("최소급여"),
    max("salary").alias("최대급여"),
    sum("salary").alias("총급여")
).show()
```

</details>

#### 실습 1-3: 고유값 개수

hire_year의 고유한 값이 몇 개인지 조회하세요.

<details>
<summary>힌트</summary>

`countDistinct("컬럼")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.agg(countDistinct("hire_year").alias("입사년도수")).show()
```

</details>

---

## Part 2: 그룹별 집계 (groupBy + agg)

### groupBy 패턴

```python
df.groupBy("그룹컬럼")                    # 단일 컬럼
df.groupBy("컬럼1", "컬럼2")              # 여러 컬럼
df.groupBy(col("컬럼"))                   # col() 사용
```

### agg 내 집계 함수

```python
.agg(
    count("*").alias("별칭1"),           # alias로 결과 컬럼명 지정
    sum("컬럼").alias("별칭2"),
    avg("컬럼").alias("별칭3"),
)
```

In [ ]:
# -----------------------------------------------------------------------------
# groupBy() + agg(): 부서별 집계
# -----------------------------------------------------------------------------

# groupBy(컬럼): 해당 컬럼 값이 같은 행들을 그룹화
# agg(): 각 그룹에 대해 집계 함수 적용

# 부서별 통계
dept_stats = df_emp.groupBy("department").agg(
    count("*").alias("인원수"),                           # 부서별 인원
    spark_round(avg("salary"), 2).alias("평균급여"),      # 평균 급여 (소수 2자리)
    min("salary").alias("최소급여"),                      # 최소 급여
    max("salary").alias("최대급여"),                      # 최대 급여
    sum("salary").alias("총급여"),                        # 급여 합계
)

print("=== 부서별 통계 ===")
dept_stats.orderBy(col("인원수").desc()).show()

In [ ]:
# -----------------------------------------------------------------------------
# groupBy(): 여러 컬럼으로 그룹화
# -----------------------------------------------------------------------------

# 여러 컬럼을 쉼표로 나열하면 조합별로 그룹화
# (부서, 입사년도) 조합별 통계

dept_year_stats = df_emp.groupBy("department", "hire_year").agg(
    count("*").alias("인원수"),
    spark_round(avg("salary"), 0).alias("평균급여"),
)

print("=== 부서 + 입사년도별 통계 ===")
dept_year_stats.orderBy("department", "hire_year").show(15)

In [ ]:
# -----------------------------------------------------------------------------
# groupBy(): 간단한 집계 shortcut
# -----------------------------------------------------------------------------

# groupBy 후 바로 count(), sum() 등 호출 가능 (단일 집계)
# agg() 생략 가능

# 부서별 인원 수 (간단 버전)
df_emp.groupBy("department").count().show()

# 부서별 급여 합계 (간단 버전)
df_emp.groupBy("department").sum("salary").show()

# 부서별 평균 급여 (간단 버전)
df_emp.groupBy("department").avg("salary").show()

In [ ]:
# -----------------------------------------------------------------------------
# 조건부 집계: when과 결합
# -----------------------------------------------------------------------------

# when을 사용해 조건에 맞는 행만 집계
# SQL의 COUNT(CASE WHEN ... THEN 1 END)와 동일

# 부서별 고연봉자(8만 이상) 수
conditional_count = df_emp.groupBy("department").agg(
    count("*").alias("전체인원"),
    # when 조건이 참인 경우만 카운트
    count(when(col("salary") >= 80000, 1)).alias("고연봉자수"),
    count(when(col("salary") < 50000, 1)).alias("저연봉자수"),
    count(when(col("age") >= 40, 1)).alias("40대이상"),
)

print("=== 조건부 집계 ===")
conditional_count.show()

### 실습

#### 실습 2-1: 부서별 인원수

부서별 직원 수를 조회하세요.

<details>
<summary>힌트</summary>

`df.groupBy("컬럼").count()`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department").count().show()
```

</details>

#### 실습 2-2: 부서별 급여 통계

부서별로 평균 급여와 최고 급여를 조회하세요.

<details>
<summary>힌트</summary>

`groupBy().agg(avg(), max())`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department").agg(
    avg("salary").alias("평균급여"),
    max("salary").alias("최고급여")
).show()
```

</details>

#### 실습 2-3: 입사년도별 인원

hire_year별로 입사한 직원 수를 조회하고, 입사년도 순으로 정렬하세요.

<details>
<summary>힌트</summary>

`groupBy().count().orderBy()`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("hire_year").count().orderBy("hire_year").show()
```

</details>

#### 실습 2-4: 부서+입사년도 조합

부서와 입사년도 조합별 인원수를 조회하세요.

<details>
<summary>힌트</summary>

`groupBy("컬럼1", "컬럼2")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department", "hire_year").count().orderBy("department", "hire_year").show()
```

</details>

---

## Part 3: 특수 집계 함수

### 리스트/집합 수집

| 함수 | 설명 | 결과 타입 |
|------|------|----------|
| `collect_list()` | 그룹의 모든 값을 배열로 | Array (중복 포함) |
| `collect_set()` | 그룹의 고유 값을 집합으로 | Array (중복 제거) |
| `first()` | 첫 번째 값 | 단일 값 |
| `last()` | 마지막 값 | 단일 값 |

In [ ]:
# -----------------------------------------------------------------------------
# collect_list(), collect_set(): 값들을 배열로 모으기
# -----------------------------------------------------------------------------

# collect_list(): 그룹의 모든 값을 배열로 수집 (중복 포함)
# collect_set(): 그룹의 고유 값만 배열로 수집 (중복 제거)

# 부서별 직원 이름 목록
name_list = df_emp.groupBy("department").agg(
    collect_list("name").alias("직원목록"),      # 모든 이름
    collect_set("hire_year").alias("입사년도"),  # 고유 입사년도
    count("*").alias("인원수"),
)

print("=== 부서별 직원 목록 ===")
name_list.show(truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# first(), last(): 첫 번째/마지막 값
# -----------------------------------------------------------------------------

# first(): 그룹의 첫 번째 값 반환
# last(): 그룹의 마지막 값 반환
# 주의: 정렬 없이 사용하면 결과가 비결정적

# 부서별 첫 번째 직원 (정렬 없이는 순서 보장 안 됨)
first_emp = df_emp.groupBy("department").agg(
    first("name").alias("첫번째직원"),
    first("salary").alias("해당급여"),
)

print("=== 부서별 첫 번째 직원 ===")
first_emp.show()

### 실습

#### 실습 3-1: collect_list 사용

부서별로 직원 이름을 배열로 수집하세요.

<details>
<summary>힌트</summary>

`groupBy().agg(collect_list("컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department").agg(
    collect_list("name").alias("직원목록")
).show(truncate=False)
```

</details>

#### 실습 3-2: collect_set 사용

부서별로 고유한 입사년도를 배열로 수집하세요.

<details>
<summary>힌트</summary>

`collect_set()`은 중복 제거됨

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department").agg(
    collect_set("hire_year").alias("입사년도목록")
).show(truncate=False)
```

</details>

#### 실습 3-3: first 함수

부서별로 첫 번째 직원의 이름과 급여를 조회하세요.

<details>
<summary>힌트</summary>

`first("컬럼")` 사용

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.groupBy("department").agg(
    first("name").alias("첫번째직원"),
    first("salary").alias("급여")
).show()
```

</details>

---

## Part 4: 조인 (Join)

### 조인 타입 비교

```
df1 (직원)              df2 (부서)
┌────────────────┐     ┌────────────────┐
│ name │ dept    │     │ dept │ head    │
├────────────────┤     ├────────────────┤
│ Kim  │ Eng     │     │ Eng  │ Alice   │
│ Lee  │ Sales   │     │ Sales│ Bob     │
│ Park │ HR      │     │ Legal│ Charlie │  ← 직원 없음
│ Choi │ IT      │     └────────────────┘
└────────────────┘       ↑ IT 부서 없음
```

| 조인 타입 | 결과 | SQL 비유 |
|-----------|------|----------|
| `inner` | 양쪽에 모두 있는 것만 | INNER JOIN |
| `left` | 왼쪽 전체 + 오른쪽 매칭 | LEFT OUTER JOIN |
| `right` | 오른쪽 전체 + 왼쪽 매칭 | RIGHT OUTER JOIN |
| `outer` | 양쪽 전체 | FULL OUTER JOIN |
| `left_semi` | 왼쪽 중 매칭되는 것만 (왼쪽 컬럼만) | WHERE EXISTS |
| `left_anti` | 왼쪽 중 매칭 안 되는 것만 | WHERE NOT EXISTS |

In [ ]:
# -----------------------------------------------------------------------------
# 조인 데이터 확인
# -----------------------------------------------------------------------------

print("=== 직원 데이터 ===")
df_emp.select("emp_id", "name", "department").show(5)

print("=== 부서 데이터 ===")
df_dept.show()

In [ ]:
# -----------------------------------------------------------------------------
# Inner Join: 양쪽에 모두 있는 것만
# -----------------------------------------------------------------------------

# join(df2, 조건, how="inner")
# - df2: 조인할 DataFrame
# - 조건: 조인 키 (문자열 또는 조건식)
# - how: 조인 타입 (기본값 "inner")

# 직원과 부서 정보 조인
# department(직원) == dept_name(부서)인 행만 결합
df_joined = df_emp.join(
    df_dept,                                    # 조인할 DataFrame
    df_emp.department == df_dept.dept_name,    # 조인 조건
    "inner"                                     # 조인 타입
)

print("=== Inner Join 결과 ===")
df_joined.select(
    "name", "department", "salary", "dept_head", "location"
).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 조인 키가 같은 이름일 때: 문자열로 지정
# -----------------------------------------------------------------------------

# 조인 키 컬럼명이 같으면 문자열로 간단히 지정
# 결과에서 조인 키 컬럼이 하나만 남음 (중복 제거)

# 데이터 준비: 컬럼명 맞추기
df_dept_renamed = df_dept.withColumnRenamed("dept_name", "department")

# 같은 이름으로 조인
df_simple_join = df_emp.join(
    df_dept_renamed,
    "department",       # 양쪽에 같은 이름의 컬럼
    "inner"
)

print("=== 같은 컬럼명으로 조인 ===")
df_simple_join.select("name", "department", "salary", "dept_head").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# Left Join: 왼쪽 전체 + 오른쪽 매칭
# -----------------------------------------------------------------------------

# left join: 왼쪽 DataFrame의 모든 행 유지
# 오른쪽에 매칭되는 행이 없으면 NULL

# IT 부서 직원 추가 (df_dept에는 IT 부서 없음)
df_emp_with_it = df_emp.union(
    spark.createDataFrame([
        ("E999", "IT_Person", "IT", 70000, 30, 2024)
    ], df_emp.columns)
)

# Left Join: 모든 직원 + 부서 정보 (없으면 NULL)
df_left = df_emp_with_it.join(
    df_dept,
    df_emp_with_it.department == df_dept.dept_name,
    "left"
)

print("=== Left Join 결과 (IT 부서 직원 포함) ===")
df_left.filter(col("dept_name").isNull()).select(
    "name", "department", "dept_name", "dept_head"
).show()

In [ ]:
# -----------------------------------------------------------------------------
# Left Semi Join: 존재 여부만 확인 (왼쪽 컬럼만 반환)
# -----------------------------------------------------------------------------

# left_semi: 오른쪽에 매칭되는 왼쪽 행만 반환
# 오른쪽 컬럼은 결과에 포함되지 않음
# SQL의 WHERE EXISTS와 동일

# 부서 정보가 있는 직원만 (부서 컬럼 없이)
df_exists = df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "left_semi"
)

print("=== Left Semi Join (존재하는 부서의 직원만) ===")
print(f"컬럼: {df_exists.columns}")
df_exists.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# Left Anti Join: 매칭 안 되는 것만 (NOT EXISTS)
# -----------------------------------------------------------------------------

# left_anti: 오른쪽에 매칭되지 않는 왼쪽 행만 반환
# SQL의 WHERE NOT EXISTS와 동일

# IT 직원 포함 데이터로 테스트
# 부서 정보가 없는 직원 찾기
df_not_exists = df_emp_with_it.join(
    df_dept,
    df_emp_with_it.department == df_dept.dept_name,
    "left_anti"
)

print("=== Left Anti Join (부서 정보 없는 직원) ===")
df_not_exists.show()

In [ ]:
# -----------------------------------------------------------------------------
# Broadcast Join: 작은 테이블 최적화
# -----------------------------------------------------------------------------

from pyspark.sql.functions import broadcast

# broadcast(): 작은 DataFrame을 모든 워커에 복사
# 대용량 + 소용량 조인 시 성능 향상
# 셔플 없이 각 워커에서 로컬 조인

# 부서 테이블은 작으므로 broadcast 적용
df_broadcast = df_emp.join(
    broadcast(df_dept),                        # 작은 테이블에 broadcast 적용
    df_emp.department == df_dept.dept_name,
    "inner"
)

print("=== Broadcast Join ===")
df_broadcast.select("name", "department", "dept_head").show(5)

### 실습

#### 실습 4-1: Inner Join

df_emp와 df_dept를 department == dept_name 조건으로 inner join하고, name, department, dept_head 컬럼을 조회하세요.

<details>
<summary>힌트</summary>

`df1.join(df2, df1.컬럼 == df2.컬럼, "inner")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "inner"
).select("name", "department", "dept_head").show(10)
```

</details>

#### 실습 4-2: Left Join

df_emp와 df_dept를 left join하여 부서 정보가 없는 직원도 포함시키세요.

<details>
<summary>힌트</summary>

`"left"` 조인 타입 사용

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "left"
).select("name", "department", "dept_head", "location").show(10)
```

</details>

#### 실습 4-3: 같은 컬럼명으로 조인

df_dept의 dept_name을 department로 이름 변경 후, 같은 컬럼명으로 조인하세요.

<details>
<summary>힌트</summary>

`withColumnRenamed()` 후 문자열로 조인

</details>

<details>
<summary>모범 답안</summary>

```python
df_dept_renamed = df_dept.withColumnRenamed("dept_name", "department")
df_emp.join(df_dept_renamed, "department", "inner").show(5)
```

</details>

#### 실습 4-4: Broadcast Join

df_dept에 broadcast를 적용하여 조인하세요.

<details>
<summary>힌트</summary>

`broadcast(작은_df)` 함수 사용

</details>

<details>
<summary>모범 답안</summary>

```python
from pyspark.sql.functions import broadcast

df_emp.join(
    broadcast(df_dept),
    df_emp.department == df_dept.dept_name,
    "inner"
).select("name", "department", "dept_head").show(5)
```

</details>

---

## Part 5: 피벗 (Pivot)

### Pivot이란?

```
원본 데이터:                    피벗 후:
┌──────┬──────┬───────┐       ┌──────┬─────┬─────┬─────┐
│ year │ qtr  │ sales │       │ year │ Q1  │ Q2  │ Q3  │
├──────┼──────┼───────┤  →    ├──────┼─────┼─────┼─────┤
│ 2024 │ Q1   │ 100   │       │ 2024 │ 100 │ 150 │ 200 │
│ 2024 │ Q2   │ 150   │       │ 2025 │ 120 │ 180 │ ... │
│ 2024 │ Q3   │ 200   │       └──────┴─────┴─────┴─────┘
│ 2025 │ Q1   │ 120   │
│ 2025 │ Q2   │ 180   │       행 → 컬럼으로 변환
└──────┴──────┴───────┘
```

In [ ]:
# -----------------------------------------------------------------------------
# pivot(): 행을 컬럼으로 변환
# -----------------------------------------------------------------------------

# 매출 데이터로 피벗 테이블 생성
# 지역별, 제품별 매출 합계

# pivot(컬럼): 해당 컬럼의 고유값들이 새 컬럼명이 됨
# pivot(컬럼, [값목록]): 특정 값만 피벗 (성능 향상)

pivot_sales = df_sales.groupBy("region").pivot(
    "product",              # 피벗할 컬럼 (A, B, C가 컬럼명이 됨)
    ["A", "B", "C"]         # 피벗할 값 목록 (명시하면 성능 향상)
).sum("amount")             # 집계 함수

print("=== 지역별 제품별 매출 (피벗) ===")
pivot_sales.show()

In [ ]:
# -----------------------------------------------------------------------------
# pivot(): 여러 집계
# -----------------------------------------------------------------------------

# 피벗 + 여러 집계 함수
# agg() 안에 여러 집계 지정

pivot_multi = df_sales.groupBy("region").pivot(
    "product", ["A", "B", "C"]
).agg(
    sum("amount").alias("총매출"),
    avg("amount").alias("평균매출"),
)

print("=== 피벗 + 여러 집계 ===")
pivot_multi.show()

### 실습

#### 실습 5-1: 기본 피벗

df_sales에서 지역(region)별로 제품(product)을 피벗하여 매출(amount) 합계를 조회하세요.

<details>
<summary>힌트</summary>

`groupBy("행그룹").pivot("피벗컬럼").sum("집계컬럼")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales.groupBy("region").pivot("product").sum("amount").show()
```

</details>

#### 실습 5-2: 피벗 값 지정

피벗 시 product 값을 ["A", "B"]만 지정하여 조회하세요.

<details>
<summary>힌트</summary>

`pivot("컬럼", [값목록])`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales.groupBy("region").pivot("product", ["A", "B"]).sum("amount").show()
```

</details>

#### 실습 5-3: 피벗 + 평균

지역별로 제품을 피벗하여 평균 매출을 조회하세요.

<details>
<summary>힌트</summary>

`pivot().avg()`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales.groupBy("region").pivot("product", ["A", "B", "C"]).avg("amount").show()
```

</details>

---

## Part 6: Spark SQL 연동

### DataFrame ↔ SQL

```python
# 1. DataFrame을 SQL 테이블로 등록
df.createOrReplaceTempView("테이블명")

# 2. SQL 쿼리 실행 → DataFrame 반환
result = spark.sql("SELECT * FROM 테이블명")
```

### 💡 View(뷰)란?
View는 "가상의 테이블"입니다.
- 실제로 데이터를 저장하지 않고, 원본 데이터를 바라보는 "창문" 같은 역할
- 마치 TV 화면처럼, 화면(View) 자체에 영상이 저장된 게 아니라
  원본 소스(DataFrame)를 보여주는 것

왜 View를 사용할까요?
```text
DataFrame (Python 문법): `df.filter(...).select(...)`
View (SQL 문법): `SELECT ... FROM ... WHERE ...`
```

- SQL에 익숙한 분석가들이 DataFrame을 SQL로 조회할 수 있게 해줍니다
- 복잡한 Python 코드 없이 익숙한 SQL로 데이터 탐색 가능

실제 테이블 vs 임시 뷰(Temp View)
- 실제 테이블: 데이터가 디스크에 물리적으로 저장됨
- 임시 뷰
    - 데이터 저장 X, 원본 DataFrame을 참조만 함
    - SparkSession 종료 시 자동으로 사라짐

In [ ]:
# -----------------------------------------------------------------------------
# createOrReplaceTempView(): SQL 테이블 등록
# -----------------------------------------------------------------------------
#
# createOrReplaceTempView(이름): 임시 뷰로 등록
# - 해당 SparkSession 내에서만 유효
# - 세션 종료 시 자동 삭제
# - 같은 이름의 뷰가 있으면 덮어쓰기 (Replace)

# DataFrame을 SQL 테이블(뷰)로 등록
df_emp.createOrReplaceTempView("employees")
df_dept.createOrReplaceTempView("departments")

print("SQL 테이블 등록 완료: employees, departments")

# -----------------------------------------------------------------------------
# 등록 후 사용 예시
# -----------------------------------------------------------------------------
# 이제 SQL 문법으로 DataFrame을 조회할 수 있습니다!
# spark.sql("SELECT * FROM employees WHERE salary > 5000")
# spark.sql("SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id")

In [ ]:
# -----------------------------------------------------------------------------
# spark.sql(): SQL 쿼리 실행
# -----------------------------------------------------------------------------

# spark.sql(쿼리문): SQL 실행 후 DataFrame 반환
# 복잡한 로직을 SQL로 작성 가능

# SQL로 부서별 통계
sql_result = spark.sql("""
    SELECT
        department,
        COUNT(*) as `인원수`,
        ROUND(AVG(salary), 2) as `평균급여`,
        MAX(salary) as `최고급여`
    FROM employees
    GROUP BY department
    ORDER BY `인원수` DESC
""")

print("=== SQL 쿼리 결과 ===")
sql_result.show()

In [ ]:
# -----------------------------------------------------------------------------
# SQL: 조인
# -----------------------------------------------------------------------------

# SQL로 조인 쿼리
sql_join = spark.sql("""
    SELECT
        e.name,
        e.department,
        e.salary,
        d.dept_head,
        d.location
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    WHERE e.salary >= 70000
    ORDER BY e.salary DESC
""")

print("=== SQL 조인 결과 ===")
sql_join.show(10)

In [ ]:
# -----------------------------------------------------------------------------
# expr(): SQL 표현식을 DataFrame에서 사용
# -----------------------------------------------------------------------------

# expr(): SQL 표현식을 Column으로 변환
# select(), withColumn() 등에서 SQL 문법 사용 가능

# SQL 표현식으로 새 컬럼 추가
df_with_expr = df_emp.withColumn(
    "salary_grade",
    expr("CASE WHEN salary >= 80000 THEN 'High' ELSE 'Normal' END")
).withColumn(
    "bonus",
    expr("salary * 0.1")  # 급여의 10%
)

print("=== expr() 사용 예시 ===")
df_with_expr.select("name", "salary", "salary_grade", "bonus").show(5)

### 실습

#### 실습 6-1: 임시 뷰 등록

df_emp를 "emp"라는 이름의 임시 뷰로 등록하세요.

<details>
<summary>힌트</summary>

`df.createOrReplaceTempView("뷰이름")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.createOrReplaceTempView("emp")
```

</details>

#### 실습 6-2: SQL 쿼리 실행

SQL로 employees 테이블에서 salary가 70000 이상인 직원의 name과 salary를 조회하세요.

<details>
<summary>힌트</summary>

`spark.sql("SELECT ... FROM ... WHERE ...")`

</details>

<details>
<summary>모범 답안</summary>

```python
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE salary >= 70000
""").show()
```

</details>

#### 실습 6-3: SQL 집계

SQL로 부서별 평균 급여를 조회하세요.

<details>
<summary>힌트</summary>

`GROUP BY` 사용

</details>

<details>
<summary>모범 답안</summary>

```python
spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
""").show()
```

</details>

#### 실습 6-4: expr 사용

expr()을 사용하여 salary의 5%를 "bonus" 컬럼으로 추가하세요.

<details>
<summary>힌트</summary>

`withColumn("컬럼", expr("SQL표현식"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df_emp.withColumn("bonus", expr("salary * 0.05")).select("name", "salary", "bonus").show(5)
```

</details>

---

## 퀴즈

Q1. 다음 코드의 결과로 올바른 것은?

```python
df.groupBy("dept").agg(count("*").alias("cnt"))
```

- A) 전체 행 수를 계산한다
- B) 부서별 NULL이 아닌 행 수를 계산한다
- C) 부서별 전체 행 수를 계산한다
- D) 에러가 발생한다

<details>
<summary>정답 보기</summary>

**정답: C) 부서별 전체 행 수를 계산한다**

`count("*")`은 NULL 포함 모든 행을 카운트합니다.
`count("특정컬럼")`은 해당 컬럼이 NULL이 아닌 행만 카운트합니다.

</details>

---

Q2. 왼쪽 DataFrame의 모든 행을 유지하면서 오른쪽과 조인하려면?

- A) `df1.join(df2, 조건, "inner")`
- B) `df1.join(df2, 조건, "left")`
- C) `df1.join(df2, 조건, "right")`
- D) `df1.join(df2, 조건, "outer")`

<details>
<summary>정답 보기</summary>

**정답: B) `df1.join(df2, 조건, "left")`**

left join은 왼쪽 DataFrame의 모든 행을 유지하고,
매칭되지 않는 행은 오른쪽 컬럼이 NULL로 채워집니다.

</details>

---

Q3. 작은 테이블을 조인할 때 성능을 높이는 방법은?

- A) `cache()`로 캐싱한다
- B) `repartition()`으로 파티션을 늘린다
- C) `broadcast()`를 사용한다
- D) `coalesce()`로 파티션을 줄인다

<details>
<summary>정답 보기</summary>

**정답: C) `broadcast()`를 사용한다**

`broadcast(df)`는 작은 DataFrame을 모든 워커 노드에 복사하여
셔플 없이 로컬에서 조인할 수 있게 합니다.

</details>

---

Q4. pivot()의 역할은?

- A) 행과 열을 바꾼다
- B) 특정 컬럼의 고유값을 새 컬럼명으로 변환한다
- C) 데이터를 정렬한다
- D) 중복을 제거한다

<details>
<summary>정답 보기</summary>

**정답: B) 특정 컬럼의 고유값을 새 컬럼명으로 변환한다**

pivot은 행 데이터를 컬럼으로 변환하여 크로스탭 형태로 만듭니다.
예: product 컬럼의 [A, B, C] 값이 컬럼명 A, B, C가 됩니다.

</details>

---

## 과제: 영업팀 월간 실적 대시보드

### 시나리오

영업팀에서 **월간 실적 대시보드**를 위한 데이터를 요청했습니다.
직원 데이터와 부서 데이터를 결합하고, 매출 데이터를 다양한 관점에서 집계하여
경영진 보고용 분석 리포트를 생성하세요.

**요구사항:**
- 직원과 부서 정보를 조인하여 통합 데이터 구성
- 지역별/제품별 매출 집계
- 목표 달성 여부 분석
- 피벗 테이블로 크로스탭 생성
- SQL과 DataFrame API 비교

---

### Step 1: 직원-부서 조인

`df_emp`와 `df_dept`를 조인하여 직원의 부서 상세 정보(부서장, 위치, 예산)를 포함한 통합 DataFrame을 생성하세요.

조인 조건: `df_emp.department == df_dept.dept_name`

결과 컬럼: `name`, `department`, `salary`, `dept_head`, `location`, `budget`

<details>
<summary>힌트</summary>

- `df1.join(df2, 조건, "inner")` 사용
- `select()`로 필요한 컬럼만 선택

</details>

<details>
<summary>모범 답안</summary>

```python
df_step1 = df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "inner"
).select(
    "name", "department", "salary", "dept_head", "location", "budget"
)

print("=== 직원-부서 통합 데이터 ===")
df_step1.show(10)
print(f"조인 결과: {df_step1.count()}명")
```

</details>

---

### Step 2: 지역별/제품별 매출 집계

`df_sales`를 사용하여 다음을 계산하세요:

1. **지역별 매출 현황**: 지역(region)별 총매출, 평균매출, 거래건수
2. **제품별 매출 현황**: 제품(product)별 총매출, 최대 단일 거래액

결과를 총매출 내림차순으로 정렬하세요.

<details>
<summary>힌트</summary>

- `groupBy().agg(sum(), avg(), count(), max())`
- `orderBy(col().desc())`

</details>

<details>
<summary>모범 답안</summary>

```python
# 지역별 매출 현황
region_sales = df_sales.groupBy("region").agg(
    sum("amount").alias("총매출"),
    spark_round(avg("amount"), 2).alias("평균매출"),
    count("*").alias("거래건수")
).orderBy(col("총매출").desc())

print("=== 지역별 매출 현황 ===")
region_sales.show()

# 제품별 매출 현황
product_sales = df_sales.groupBy("product").agg(
    sum("amount").alias("총매출"),
    max("amount").alias("최대거래액")
).orderBy(col("총매출").desc())

print("=== 제품별 매출 현황 ===")
product_sales.show()
```

</details>

---

### Step 3: 조건부 집계 - 목표 달성 분석

매출 목표가 500이라고 가정할 때, 지역별로 다음을 집계하세요:
- 전체 거래 건수
- 목표 달성 건수 (amount >= 500)
- 목표 미달 건수 (amount < 500)
- 달성률 (%) = 목표 달성 건수 / 전체 건수 × 100

<details>
<summary>힌트</summary>

- `count(when(조건, 1))` 패턴 사용
- 달성률 계산 시 `spark_round()`로 소수점 처리

</details>

<details>
<summary>모범 답안</summary>

```python
target_analysis = df_sales.groupBy("region").agg(
    count("*").alias("전체건수"),
    count(when(col("amount") >= 500, 1)).alias("달성건수"),
    count(when(col("amount") < 500, 1)).alias("미달건수"),
    spark_round(
        count(when(col("amount") >= 500, 1)) / count("*") * 100, 1
    ).alias("달성률")
)

print("=== 목표 달성 분석 (목표: 500) ===")
target_analysis.orderBy(col("달성률").desc()).show()
```

</details>

---

### Step 4: 피벗 테이블 생성

지역(행) × 제품(열) 형태의 피벗 테이블을 만들어 각 조합의 총매출을 한눈에 볼 수 있게 하세요.

<details>
<summary>힌트</summary>

`groupBy("행그룹").pivot("열그룹", [값목록]).sum("집계컬럼")`

</details>

<details>
<summary>모범 답안</summary>

```python
pivot_table = df_sales.groupBy("region").pivot(
    "product", ["A", "B", "C"]
).sum("amount")

print("=== 지역 × 제품 매출 피벗 테이블 ===")
pivot_table.show()
```

</details>

---

### Step 5: SQL로 종합 분석

Spark SQL을 사용하여 다음 분석을 수행하세요:

1. 테이블 등록: `df_sales`를 "sales"로, `df_emp`를 "employees"로 등록
2. SQL로 지역별 매출 Top 3 조회
3. SQL로 제품별 평균 매출이 400 이상인 제품만 조회

<details>
<summary>힌트</summary>

- `createOrReplaceTempView()`
- `spark.sql()` 사용
- `HAVING` 절로 집계 결과 필터링

</details>

<details>
<summary>모범 답안</summary>

```python
# 테이블 등록
df_sales.createOrReplaceTempView("sales")
df_emp.createOrReplaceTempView("employees")

# 지역별 매출 Top 3
print("=== [SQL] 지역별 매출 Top 3 ===")
spark.sql("""
    SELECT region, SUM(amount) as total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
    LIMIT 3
""").show()

# 평균 매출 400 이상 제품
print("=== [SQL] 평균 매출 400 이상 제품 ===")
spark.sql("""
    SELECT product, AVG(amount) as avg_sales
    FROM sales
    GROUP BY product
    HAVING AVG(amount) >= 400
    ORDER BY avg_sales DESC
""").show()
```

</details>

---

### 최종 과제: 종합 대시보드 데이터 생성

아래 요구사항을 모두 만족하는 **대시보드용 요약 DataFrame**을 생성하세요:

1. `df_sales`와 가상의 지역 정보를 조인 (Seoul=본사, Busan=지사, Daegu=지사)
2. 지역별로 집계: 총매출, 평균매출, 거래건수, 목표달성률
3. "본사/지사" 구분 컬럼 추가
4. 총매출 내림차순 정렬

<details>
<summary>힌트</summary>

- 지역 정보 DataFrame을 직접 생성
- 조인 후 집계
- `when()`으로 본사/지사 구분

</details>

<details>
<summary>모범 답안</summary>

```python
# 지역 정보 DataFrame 생성
region_info = spark.createDataFrame([
    ("Seoul", "본사"),
    ("Busan", "지사"),
    ("Daegu", "지사")
], ["region", "지역구분"])

# 집계 후 조인
dashboard = (
    df_sales
    .groupBy("region")
    .agg(
        sum("amount").alias("총매출"),
        spark_round(avg("amount"), 2).alias("평균매출"),
        count("*").alias("거래건수"),
        spark_round(count(when(col("amount") >= 500, 1)) / count("*") * 100, 1).alias("달성률")
    )
    .join(region_info, "region", "left")
    .select("region", "지역구분", "총매출", "평균매출", "거래건수", "달성률")
    .orderBy(col("총매출").desc())
)

print("=== 영업 대시보드 요약 ===")
dashboard.show()
```

</details>

---

## 핵심 요약

### 전체 집계

```python
df.agg(
    count("*").alias("총건수"),
    sum("amount").alias("총합"),
    avg("amount").alias("평균"),
)
```

### 그룹별 집계

```python
df.groupBy("컬럼1", "컬럼2").agg(
    count("*").alias("건수"),
    sum("금액").alias("합계"),
)
```

### 조인

| 조인 타입 | 코드 |
|-----------|------|
| Inner | `df1.join(df2, 조건, "inner")` |
| Left | `df1.join(df2, 조건, "left")` |
| Broadcast | `df1.join(broadcast(df2), 조건)` |

### 피벗

```python
df.groupBy("행").pivot("열", [값목록]).sum("집계컬럼")
```

### SQL 연동

```python
df.createOrReplaceTempView("테이블명")
spark.sql("SELECT * FROM 테이블명")
```

In [ ]:
# 세션 유지 (다음 교시 계속)
print("\n3교시 완료! 다음 교시: 실무 핵심 패턴")

# Day 12 - 3교시: 실무 핵심 패턴

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- NULL 값을 효과적으로 처리할 수 있다 (dropna, fillna, coalesce)
- 문자열/날짜 함수를 활용할 수 있다
- Window 함수로 순위, 누적합, 이동평균을 계산할 수 있다
- 실무에서 자주 사용하는 패턴들을 적용할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when,
    # NULL 처리
    coalesce, isnan,
    # 문자열 함수
    concat, concat_ws, substring, length, trim, ltrim, rtrim,
    upper, lower, initcap, regexp_replace, regexp_extract, split,
    lpad, rpad,
    # 날짜/시간 함수
    current_date, current_timestamp, to_date, to_timestamp, date_format,
    year, month, dayofmonth, dayofweek, hour, minute,
    date_add, date_sub, datediff, months_between, trunc,
    # 집계/윈도우
    count, sum, avg, min, max, first, last,
    row_number, rank, dense_rank, lag, lead,
    # 기타
    round as spark_round, expr,
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Advanced-Patterns") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

os.makedirs("/tmp/spark_tutorial", exist_ok=True)
np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터: 다양한 상황을 포함한 직원 데이터
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 31)],
    "name": [f"  Employee {i}  " for i in range(1, 31)],  # 앞뒤 공백
    "email": [f"emp{i}@company.com" for i in range(1, 31)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", None], 30
    ),
    "salary": [
        50000, None, 70000, 80000, None,  # NULL 포함
        60000, 90000, 55000, None, 75000,
        65000, 85000, None, 72000, 68000,
        None, 95000, 62000, 78000, None,
        58000, 82000, 67000, None, 73000,
        69000, 88000, None, 76000, 71000
    ],
    "join_date": pd.date_range("2020-01-15", periods=30, freq="45D").strftime("%Y-%m-%d"),
})
employees.to_csv("/tmp/spark_tutorial/employees_advanced.csv", index=False)

# 매출 시계열 데이터
sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=90, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 90),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 90),
    "amount": np.random.randint(100, 1000, 90),
})
sales.to_csv("/tmp/spark_tutorial/sales_ts.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees_advanced.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales_ts.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df.count()}명, 매출: {df_sales.count()}건")
df.show(10)

---

## Part 1: NULL 처리

### NULL 관련 메서드

| 메서드 | 설명 | 예시 |
|--------|------|------|
| `isNull()` | NULL 여부 확인 | `col("x").isNull()` |
| `isNotNull()` | NULL이 아닌지 확인 | `col("x").isNotNull()` |
| `dropna()` | NULL 행 제거 | `df.dropna()` |
| `fillna()` | NULL을 다른 값으로 채우기 | `df.fillna(0)` |
| `coalesce()` | 첫 번째 non-null 값 반환 | `coalesce(col1, col2)` |

In [ ]:
# -----------------------------------------------------------------------------
# NULL 확인
# -----------------------------------------------------------------------------

# isNull(), isNotNull(): NULL 여부 확인
print("=== NULL인 행 (salary) ===")
df.filter(col("salary").isNull()).select("emp_id", "name", "salary").show()

print("=== NULL이 아닌 행 (salary) ===")
df.filter(col("salary").isNotNull()).count()

# NULL 개수 세기
null_counts = df.agg(
    count(when(col("salary").isNull(), 1)).alias("salary_null"),
    count(when(col("department").isNull(), 1)).alias("dept_null"),
)
print("=== NULL 개수 ===")
null_counts.show()

In [ ]:
# -----------------------------------------------------------------------------
# dropna(): NULL 행 제거
# -----------------------------------------------------------------------------

# dropna(): NULL이 하나라도 있는 행 제거
# dropna(how="all"): 모든 컬럼이 NULL인 행만 제거
# dropna(subset=[]): 특정 컬럼에서만 NULL 검사

# 기본: 어떤 컬럼이든 NULL이면 제거
df_no_null = df.dropna()
print(f"=== dropna() 후 행 수: {df_no_null.count()} (원본: {df.count()}) ===")

# 특정 컬럼에 NULL이 있는 행만 제거
df_no_salary_null = df.dropna(subset=["salary"])
print(f"=== salary NULL 제거 후: {df_no_salary_null.count()} ===")

# 여러 컬럼 지정
df_clean = df.dropna(subset=["salary", "department"])
print(f"=== salary, department NULL 제거 후: {df_clean.count()} ===")

In [ ]:
# -----------------------------------------------------------------------------
# fillna(): NULL을 다른 값으로 채우기
# -----------------------------------------------------------------------------

# fillna(값): 모든 컬럼의 NULL을 해당 값으로 채움
# fillna(값, subset=[]): 특정 컬럼만 채움
# fillna({"컬럼": 값}): 컬럼별로 다른 값으로 채움

# 단일 값으로 채우기 (타입 일치 필요)
df_filled_salary = df.fillna(0, subset=["salary"])
print("=== salary NULL → 0 ===")
df_filled_salary.filter(col("emp_id").isin("E002", "E005")).show()

# 컬럼별로 다른 값 채우기 (딕셔너리)
df_filled = df.fillna({
    "salary": 0,                    # 급여 NULL → 0
    "department": "Unknown"         # 부서 NULL → Unknown
})

print("=== 컬럼별 다른 값으로 채우기 ===")
df_filled.filter(
    col("emp_id").isin("E002", "E005", "E009")
).show()

In [ ]:
# -----------------------------------------------------------------------------
# coalesce(): 첫 번째 non-null 값 반환
# -----------------------------------------------------------------------------

# coalesce(col1, col2, ...): 왼쪽부터 확인하여 첫 번째 non-null 값 반환
# SQL의 COALESCE와 동일
# 여러 컬럼 중 대체 값을 찾을 때 유용

# 예시: primary_phone → secondary_phone → "연락처없음" 순서로 채우기
df_contact = spark.createDataFrame([
    ("E001", "010-1234-5678", None),
    ("E002", None, "02-555-1234"),
    ("E003", None, None),
    ("E004", "010-9999-8888", "02-111-2222"),
], ["emp_id", "primary_phone", "secondary_phone"])

# coalesce: 첫 번째 non-null 값 선택
df_with_phone = df_contact.withColumn(
    "contact",
    coalesce(
        col("primary_phone"),       # 1순위: 휴대폰
        col("secondary_phone"),     # 2순위: 유선전화
        lit("연락처없음")             # 3순위: 기본값
    )
)

print("=== coalesce 예시 ===")
df_with_phone.show()

### 실습

#### 실습 1-1: NULL 확인

salary가 NULL인 행의 개수를 조회하세요.

<details>
<summary>힌트</summary>

`filter(col("컬럼").isNull()).count()`

</details>

<details>
<summary>모범 답안</summary>

```python
df.filter(col("salary").isNull()).count()
```

</details>

#### 실습 1-2: dropna 사용

salary 컬럼에 NULL이 있는 행을 제거하세요.

<details>
<summary>힌트</summary>

`df.dropna(subset=["컬럼"])`

</details>

<details>
<summary>모범 답안</summary>

```python
df.dropna(subset=["salary"]).count()
```

</details>

#### 실습 1-3: fillna 사용

salary가 NULL인 경우 50000으로, department가 NULL인 경우 "Unknown"으로 채우세요.

<details>
<summary>힌트</summary>

`df.fillna({"컬럼1": 값1, "컬럼2": 값2})`

</details>

<details>
<summary>모범 답안</summary>

```python
df.fillna({"salary": 50000, "department": "Unknown"}).show(10)
```

</details>

#### 실습 1-4: coalesce 사용

salary가 NULL인 경우 0을 반환하는 "salary_filled" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`coalesce(col("컬럼"), lit(기본값))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("salary_filled", coalesce(col("salary"), lit(0))).show(10)
```

</details>

---

## Part 2: 문자열 함수

### 주요 문자열 함수

| 함수 | 설명 | 예시 |
|------|------|------|
| `concat()` | 문자열 합치기 | `concat(col1, col2)` |
| `concat_ws()` | 구분자로 합치기 | `concat_ws("-", col1, col2)` |
| `substring()` | 부분 문자열 | `substring(col, 1, 3)` |
| `trim()` | 앞뒤 공백 제거 | `trim(col)` |
| `upper()`/`lower()` | 대소문자 변환 | `upper(col)` |
| `split()` | 문자열 분리 → 배열 | `split(col, ",")` |
| `regexp_replace()` | 정규식 치환 | `regexp_replace(col, "패턴", "대체")` |

In [ ]:
# -----------------------------------------------------------------------------
# concat(), concat_ws(): 문자열 합치기
# -----------------------------------------------------------------------------

# concat(): 여러 문자열/컬럼을 연결
# concat_ws(구분자, ...): 구분자로 연결 (NULL 자동 건너뜀)

df_concat = df.select(
    col("emp_id"),
    col("name"),
    # concat: 단순 연결
    concat(col("emp_id"), lit("-"), col("name")).alias("id_name"),
    # concat_ws: 구분자로 연결 (NULL은 건너뜀)
    concat_ws("_", col("emp_id"), col("department")).alias("id_dept"),
)

print("=== 문자열 합치기 ===")
df_concat.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# trim(), upper(), lower(): 공백/대소문자
# -----------------------------------------------------------------------------

# trim(): 앞뒤 공백 제거
# ltrim(): 왼쪽 공백 제거
# rtrim(): 오른쪽 공백 제거
# upper(): 대문자로
# lower(): 소문자로
# initcap(): 첫 글자만 대문자

# 이름에 앞뒤 공백이 있음 → trim으로 정리
df_cleaned = df.withColumn(
    "name_trimmed",
    trim(col("name"))                    # 앞뒤 공백 제거
).withColumn(
    "name_upper",
    upper(trim(col("name")))             # 대문자 변환
).withColumn(
    "name_lower",
    lower(trim(col("name")))             # 소문자 변환
)

print("=== 공백/대소문자 처리 ===")
df_cleaned.select("name", "name_trimmed", "name_upper", "name_lower").show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# substring(): 부분 문자열
# -----------------------------------------------------------------------------

# substring(컬럼, 시작위치, 길이)
# 주의: 시작 위치는 1부터 (0이 아님!)

df_substr = df.select(
    col("email"),
    # 이메일에서 @ 앞부분 추출 (간단 버전, 정확히는 split 사용)
    substring(col("email"), 1, 4).alias("first_4_chars"),
)

print("=== substring ===")
df_substr.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# split(): 문자열 분리 → 배열
# -----------------------------------------------------------------------------

# split(컬럼, 구분자): 문자열을 배열로 분리
# 결과[0], 결과[1] 등으로 인덱싱

df_split = df.select(
    col("email"),
    # @ 기준으로 분리
    split(col("email"), "@").alias("email_parts"),
    # 배열의 첫 번째 요소 (사용자명)
    split(col("email"), "@")[0].alias("username"),
    # 배열의 두 번째 요소 (도메인)
    split(col("email"), "@")[1].alias("domain"),
)

print("=== split ===")
df_split.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# regexp_replace(), regexp_extract(): 정규식
# -----------------------------------------------------------------------------

# regexp_replace(컬럼, 패턴, 대체문자): 패턴 매칭 부분 치환
# regexp_extract(컬럼, 패턴, 그룹번호): 패턴 매칭 부분 추출

df_regex = df.select(
    col("name"),
    col("email"),
    # 숫자만 추출 (emp1 → 1)
    regexp_extract(col("email"), r"emp(\d+)", 1).alias("emp_number"),
    # 특수문자 제거
    regexp_replace(col("email"), r"@.*", "").alias("without_domain"),
)

print("=== 정규식 ===")
df_regex.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# lpad(), rpad(): 패딩 (자릿수 맞추기)
# -----------------------------------------------------------------------------

# lpad(컬럼, 총길이, 채울문자): 왼쪽 패딩
# rpad(컬럼, 총길이, 채울문자): 오른쪽 패딩

df_pad = df.select(
    col("emp_id"),
    # 사원번호를 10자리로 (앞에 0 채움)
    lpad(col("emp_id"), 10, "0").alias("padded_id"),
)

print("=== 패딩 ===")
df_pad.show(5)

### 실습

#### 실습 2-1: 공백 제거

name 컬럼의 앞뒤 공백을 제거한 "name_clean" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`trim(col("컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("name_clean", trim(col("name"))).select("name", "name_clean").show(5, truncate=False)
```

</details>

#### 실습 2-2: 대문자 변환

name 컬럼을 대문자로 변환한 "name_upper" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`upper(col("컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("name_upper", upper(trim(col("name")))).select("name", "name_upper").show(5)
```

</details>

#### 실습 2-3: 문자열 분리

email에서 @ 앞부분(사용자명)만 추출한 "username" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`split(col("컬럼"), "@")[0]`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("username", split(col("email"), "@")[0]).select("email", "username").show(5)
```

</details>

#### 실습 2-4: 문자열 합치기

emp_id와 name을 "-"로 연결한 "id_name" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`concat_ws("-", col1, col2)`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("id_name", concat_ws("-", col("emp_id"), trim(col("name")))).select("emp_id", "name", "id_name").show(5)
```

</details>

---

## Part 3: 날짜/시간 함수

### 주요 날짜 함수

| 함수 | 설명 | 예시 |
|------|------|------|
| `current_date()` | 오늘 날짜 | - |
| `to_date()` | 문자열 → 날짜 | `to_date(col, "yyyy-MM-dd")` |
| `date_format()` | 날짜 포맷 변경 | `date_format(col, "yyyy/MM")` |
| `year()`, `month()` | 연/월 추출 | `year(col)` |
| `datediff()` | 날짜 차이 | `datediff(end, start)` |
| `date_add()` | 날짜 더하기 | `date_add(col, 7)` |

In [ ]:
# -----------------------------------------------------------------------------
# to_date(), to_timestamp(): 문자열 → 날짜/시간 변환
# -----------------------------------------------------------------------------

# to_date(컬럼, 포맷): 문자열을 날짜로 변환
# to_timestamp(컬럼, 포맷): 문자열을 타임스탬프로 변환

df_date = df.withColumn(
    "join_date_parsed",
    to_date(col("join_date"), "yyyy-MM-dd")   # 문자열 → DateType
)

print("=== 날짜 파싱 ===")
df_date.select("join_date", "join_date_parsed").show(5)
df_date.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# year(), month(), dayofmonth(): 날짜 부분 추출
# -----------------------------------------------------------------------------

# year(): 연도 추출
# month(): 월 추출
# dayofmonth(): 일 추출
# dayofweek(): 요일 (1=일요일, 7=토요일)
# hour(), minute(), second(): 시간 부분 추출

df_parts = df_date.select(
    col("join_date_parsed"),
    year(col("join_date_parsed")).alias("year"),             # 연도
    month(col("join_date_parsed")).alias("month"),           # 월
    dayofmonth(col("join_date_parsed")).alias("day"),        # 일
    dayofweek(col("join_date_parsed")).alias("dow"),         # 요일 (1=일)
)

print("=== 날짜 부분 추출 ===")
df_parts.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# date_format(): 날짜 포맷 변경
# -----------------------------------------------------------------------------

# date_format(컬럼, 포맷): 날짜를 지정한 포맷의 문자열로 변환
# 포맷 패턴: yyyy(년), MM(월), dd(일), HH(시), mm(분), ss(초)

df_formatted = df_date.select(
    col("join_date_parsed"),
    # 다양한 포맷으로 변환
    date_format(col("join_date_parsed"), "yyyy/MM/dd").alias("slash_format"),
    date_format(col("join_date_parsed"), "yyyy-MM").alias("year_month"),
    date_format(col("join_date_parsed"), "yyyy년 MM월 dd일").alias("korean"),
    date_format(col("join_date_parsed"), "EEEE").alias("day_name"),  # 요일 이름
)

print("=== 날짜 포맷 변경 ===")
df_formatted.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# date_add(), date_sub(), datediff(): 날짜 연산
# -----------------------------------------------------------------------------

# date_add(컬럼, 일수): 날짜에 일 더하기
# date_sub(컬럼, 일수): 날짜에서 일 빼기
# datediff(end, start): 두 날짜 사이 일수

df_calc = df_date.select(
    col("emp_id"),
    col("join_date_parsed").alias("join_date"),
    # 7일 후
    date_add(col("join_date_parsed"), 7).alias("plus_7_days"),
    # 30일 전
    date_sub(col("join_date_parsed"), 30).alias("minus_30_days"),
    # 오늘까지 며칠?
    datediff(current_date(), col("join_date_parsed")).alias("days_since_join"),
)

print("=== 날짜 연산 ===")
df_calc.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# trunc(): 날짜 자르기 (월초, 연초 등)
# -----------------------------------------------------------------------------

# trunc(컬럼, "단위"): 해당 단위의 시작점으로 자름
# "month" → 해당 월의 1일
# "year" → 해당 연도의 1월 1일
# "week" → 해당 주의 월요일

df_trunc = df_date.select(
    col("join_date_parsed"),
    trunc(col("join_date_parsed"), "month").alias("month_start"),   # 월초
    trunc(col("join_date_parsed"), "year").alias("year_start"),     # 연초
)

print("=== 날짜 자르기 ===")
df_trunc.show(5)

### 실습

#### 실습 3-1: 날짜 파싱

join_date 문자열을 날짜 타입으로 변환한 "join_date_dt" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`to_date(col("컬럼"), "yyyy-MM-dd")`

</details>

<details>
<summary>모범 답안</summary>

```python
df.withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd")).show(5)
```

</details>

#### 실습 3-2: 연/월 추출

join_date에서 입사 연도(join_year)와 입사 월(join_month) 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`year(col("컬럼"))`, `month(col("컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df_date = df.withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd"))
df_date.withColumn("join_year", year(col("join_date_dt"))) \
    .withColumn("join_month", month(col("join_date_dt"))) \
    .select("join_date", "join_year", "join_month").show(5)
```

</details>

#### 실습 3-3: 근속일수 계산

오늘 날짜 기준 근속일수(tenure_days)를 계산하세요.

<details>
<summary>힌트</summary>

`datediff(current_date(), col("날짜컬럼"))`

</details>

<details>
<summary>모범 답안</summary>

```python
df_date = df.withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd"))
df_date.withColumn("tenure_days", datediff(current_date(), col("join_date_dt"))) \
    .select("emp_id", "join_date", "tenure_days").show(5)
```

</details>

#### 실습 3-4: 날짜 포맷 변경

join_date를 "yyyy년 MM월" 형식으로 변환한 "join_ym" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`date_format(col("날짜컬럼"), "포맷")`

</details>

<details>
<summary>모범 답안</summary>

```python
df_date = df.withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd"))
df_date.withColumn("join_ym", date_format(col("join_date_dt"), "yyyy년 MM월")) \
    .select("join_date", "join_ym").show(5)
```

</details>

---

## Part 4: Window 함수

### Window 함수란?

![윈도우 함수](https://blog.kakaocdn.net/dna/4hR8j/btsv8OgRNXR/AAAAAAAAAAAAAAAAAAAAAPMva-SZwR-CC-bRi63FDh6eG5bN1-RlKN9jMimPLFnt/img.png?credential=yqXZFxpELC7KVnFOS48ylbz2pIh7yKj8&expires=1769871599&allow_ip=&allow_referer=&signature=fZkyGS4S3BOefiARWzT%2Bd5YF%2FoE%3D)

```
┌─────────────────────────────────────────────────────────────────┐
│                      Window 함수 개념                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  일반 집계 (groupBy):                                          │
│  ┌─────┬────────┐    groupBy("dept")   ┌─────┬─────┐           │
│  │ dept│ salary │    ───────────────→  │ dept│ sum │           │
│  │ A   │ 100    │                      │ A   │ 300 │  행 감소  │
│  │ A   │ 200    │                      │ B   │ 500 │           │
│  │ B   │ 500    │                      └─────┴─────┘           │
│  └─────┴────────┘                                               │
│                                                                 │
│  Window 함수:                                                   │
│  ┌─────┬────────┐    Window("dept")    ┌─────┬────────┬─────┐  │
│  │ dept│ salary │    ───────────────→  │ dept│ salary │ sum │  │
│  │ A   │ 100    │                      │ A   │ 100    │ 300 │  │
│  │ A   │ 200    │                      │ A   │ 200    │ 300 │  │
│  │ B   │ 500    │                      │ B   │ 500    │ 500 │  │
│  └─────┴────────┘                      └─────┴────────┴─────┘  │
│                                                                 │
│  ★ 행 수 유지! 각 행에 그룹 집계값 추가                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Window 정의

```python
from pyspark.sql.window import Window

# 파티션(그룹) + 정렬
window_spec = Window.partitionBy("dept").orderBy("salary")

# 파티션만
window_spec = Window.partitionBy("dept")

# 범위 지정 (누적합 등)
window_spec = Window.partitionBy("dept").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

In [ ]:
# -----------------------------------------------------------------------------
# Window 기본: 순위 함수
# -----------------------------------------------------------------------------

from pyspark.sql.window import Window

# Window 정의: 부서별로 그룹화, 급여 내림차순 정렬
window_rank = Window.partitionBy("department").orderBy(col("salary").desc())

# 순위 함수 적용
# row_number(): 동점이어도 순차적 번호 (1, 2, 3, 4...)
# rank(): 동점은 같은 순위, 다음 순위 건너뜀 (1, 1, 3, 4...)
# dense_rank(): 동점은 같은 순위, 순위 안 건너뜀 (1, 1, 2, 3...)
df_ranked = df.filter(col("salary").isNotNull()).withColumn(
    "row_num", row_number().over(window_rank)
).withColumn(
    "rank", rank().over(window_rank)
).withColumn(
    "dense_rank", dense_rank().over(window_rank)
)

print("=== 부서별 급여 순위 ===")
df_ranked.select(
    "department", "name", "salary", "row_num", "rank", "dense_rank"
).orderBy("department", "row_num").show(15)

**LAG 함수**
![LAG 함수](https://learnsql.com/blog/lead-and-lag-functions-in-sql/1.webp)

**LEAD 함수**
![LEAD 함수](https://learnsql.com/blog/lead-and-lag-functions-in-sql/2.webp)

In [ ]:
# -----------------------------------------------------------------------------
# lag(), lead(): 이전/다음 행 참조
# -----------------------------------------------------------------------------

# lag(컬럼, n): n행 이전 값
# lead(컬럼, n): n행 이후 값
# 시계열 분석, 전일 대비 비교 등에 유용

# 날짜순 정렬된 매출 데이터로 실습
df_sales_parsed = df_sales.withColumn("date_parsed", to_date(col("date")))

# Window: 제품별, 날짜순
window_sales = Window.partitionBy("product").orderBy("date_parsed")

df_with_prev = df_sales_parsed.withColumn(
    "prev_amount", lag("amount", 1).over(window_sales)      # 전일 매출
).withColumn(
    "next_amount", lead("amount", 1).over(window_sales)     # 익일 매출
).withColumn(
    # 전일 대비 증감
    "change", col("amount") - col("prev_amount")
)

print("=== lag/lead: 전일 대비 비교 ===")
df_with_prev.filter(col("product") == "A").select(
    "date_parsed", "product", "amount", "prev_amount", "change"
).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 누적 합계 / 이동 평균
# -----------------------------------------------------------------------------

# 범위 지정: rowsBetween(시작, 끝)
# unboundedPreceding: 파티션 시작부터
# currentRow: 현재 행까지
# unboundedFollowing: 파티션 끝까지
# 숫자: 현재 행 기준 상대 위치 (-3, 0, 2 등)

# 누적 합계: 시작부터 현재까지
window_cumsum = Window.partitionBy("product").orderBy("date_parsed") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 7일 이동 평균: 최근 7일
window_ma7 = Window.partitionBy("product").orderBy("date_parsed") \
    .rowsBetween(-6, Window.currentRow)  # 현재 포함 7일

df_cumsum = df_sales_parsed.withColumn(
    "cumsum", sum("amount").over(window_cumsum)
).withColumn(
    "ma7", spark_round(avg("amount").over(window_ma7), 2)
)

print("=== 누적합 / 7일 이동평균 ===")
df_cumsum.filter(col("product") == "A").select(
    "date_parsed", "product", "amount", "cumsum", "ma7"
).show(15)

In [ ]:
# -----------------------------------------------------------------------------
# 실무 패턴: 그룹별 Top N
# -----------------------------------------------------------------------------

# 부서별 급여 Top 3 뽑기
# 1. row_number()로 순위 매기기
# 2. filter로 순위 <= 3 필터링

window_top = Window.partitionBy("department").orderBy(col("salary").desc())

df_top3 = df.filter(col("salary").isNotNull()).withColumn(
    "rank", row_number().over(window_top)
).filter(
    col("rank") <= 3
)

print("=== 부서별 급여 Top 3 ===")
df_top3.select("department", "name", "salary", "rank") \
    .orderBy("department", "rank").show()

### 실습

#### 실습 4-1: 순위 함수

부서별로 급여 내림차순 순위(rank)를 매긴 "salary_rank" 컬럼을 추가하세요.

<details>
<summary>힌트</summary>

`Window.partitionBy("컬럼").orderBy(col("컬럼").desc())`

</details>

<details>
<summary>모범 답안</summary>

```python
from pyspark.sql.window import Window

window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df.filter(col("salary").isNotNull()).withColumn(
    "salary_rank", row_number().over(window_spec)
).select("department", "name", "salary", "salary_rank").show(10)
```

</details>

#### 실습 4-2: lag 함수

매출 데이터에서 제품별 전일 매출(prev_amount)을 추가하세요.

<details>
<summary>힌트</summary>

`lag("컬럼", 1).over(window_spec)`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales_parsed = df_sales.withColumn("date_parsed", to_date(col("date")))
window_spec = Window.partitionBy("product").orderBy("date_parsed")

df_sales_parsed.withColumn(
    "prev_amount", lag("amount", 1).over(window_spec)
).filter(col("product") == "A").show(10)
```

</details>

#### 실습 4-3: 누적 합계

제품별 날짜순 누적 매출(cumsum)을 계산하세요.

<details>
<summary>힌트</summary>

`rowsBetween(Window.unboundedPreceding, Window.currentRow)`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales_parsed = df_sales.withColumn("date_parsed", to_date(col("date")))
window_cumsum = Window.partitionBy("product").orderBy("date_parsed") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_sales_parsed.withColumn(
    "cumsum", sum("amount").over(window_cumsum)
).filter(col("product") == "A").select("date_parsed", "amount", "cumsum").show(10)
```

</details>

#### 실습 4-4: 그룹별 Top N

부서별 급여 상위 2명만 필터링하세요.

<details>
<summary>힌트</summary>

row_number() + filter(rank <= 2)

</details>

<details>
<summary>모범 답안</summary>

```python
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

df.filter(col("salary").isNotNull()).withColumn(
    "rank", row_number().over(window_spec)
).filter(col("rank") <= 2) \
    .select("department", "name", "salary", "rank").orderBy("department", "rank").show()
```

</details>

---

## Part 5: 자주 쓰는 실무 패턴

### 패턴 모음

1. 조건부 집계
2. 전체 대비 비율
3. 데이터 품질 체크
4. 컬럼명 일괄 변경
5. Forward Fill (이전 값으로 채우기)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 1: 조건부 집계
# -----------------------------------------------------------------------------

# groupBy + when 조합으로 조건별 카운트/합계

conditional_agg = df.groupBy("department").agg(
    count("*").alias("total"),
    # 조건별 카운트
    count(when(col("salary") >= 80000, 1)).alias("high_salary_cnt"),
    count(when(col("salary") < 50000, 1)).alias("low_salary_cnt"),
    # 조건별 합계
    sum(when(col("salary") >= 80000, col("salary"))).alias("high_salary_sum"),
)

print("=== 패턴 1: 조건부 집계 ===")
conditional_agg.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 2: 전체/그룹 대비 비율
# -----------------------------------------------------------------------------

# Window 함수로 그룹 전체 합계를 각 행에 추가
window_dept = Window.partitionBy("department")

df_ratio = df.filter(col("salary").isNotNull()).withColumn(
    "dept_total", sum("salary").over(window_dept)           # 부서 총급여
).withColumn(
    "pct_of_dept", spark_round(col("salary") / col("dept_total") * 100, 2)  # 부서 내 비율
)

print("=== 패턴 2: 부서 내 급여 비율 ===")
df_ratio.select("department", "name", "salary", "dept_total", "pct_of_dept") \
    .orderBy("department", col("salary").desc()).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 3: 데이터 품질 체크
# -----------------------------------------------------------------------------

# 한 번에 여러 품질 지표 확인
quality_check = df.agg(
    count("*").alias("total_rows"),
    count("emp_id").alias("emp_id_non_null"),
    countDistinct("emp_id").alias("emp_id_unique"),
    count("salary").alias("salary_non_null"),
    sum(when(col("salary") < 0, 1).otherwise(0)).alias("negative_salary"),
    sum(when(col("department").isNull(), 1).otherwise(0)).alias("dept_null"),
)

print("=== 패턴 3: 데이터 품질 체크 ===")
quality_check.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 4: 컬럼명 일괄 변경 (소문자, 공백→언더스코어)
# -----------------------------------------------------------------------------

# toDF()로 모든 컬럼명 한번에 변경
# 리스트 컴프리헨션으로 변환 규칙 적용

# 예시 DataFrame
df_messy_cols = spark.createDataFrame([
    (1, "A", 100),
    (2, "B", 200),
], ["User ID", "Product Name", "Total Amount"])

print("변경 전:", df_messy_cols.columns)

# 소문자 + 공백을 _로 변환
new_cols = [c.lower().replace(" ", "_") for c in df_messy_cols.columns]
df_clean_cols = df_messy_cols.toDF(*new_cols)

print("변경 후:", df_clean_cols.columns)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 5: Forward Fill (이전 값으로 NULL 채우기)
# -----------------------------------------------------------------------------

# last(ignorenulls=True)를 Window와 함께 사용
# NULL을 직전 non-null 값으로 채움

df_with_null = spark.createDataFrame([
    (1, "2024-01-01", 100),
    (1, "2024-01-02", None),
    (1, "2024-01-03", None),
    (1, "2024-01-04", 200),
    (1, "2024-01-05", None),
], ["id", "date", "value"])

# Window: 시작~현재까지
window_ff = Window.partitionBy("id").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_filled = df_with_null.withColumn(
    "value_filled",
    # ignorenulls=True: NULL을 무시하고 마지막 non-null 값 반환
    last("value", ignorenulls=True).over(window_ff)
)

print("=== 패턴 5: Forward Fill ===")
df_filled.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 6: 여러 컬럼에 같은 처리 일괄 적용
# -----------------------------------------------------------------------------

# 리스트 컴프리헨션 + agg로 여러 컬럼에 같은 집계 적용
cols_to_sum = ["salary", "age"]

# 방법: 리스트 컴프리헨션으로 집계 함수 생성
agg_exprs = [sum(c).alias(f"total_{c}") for c in cols_to_sum]
agg_exprs += [avg(c).alias(f"avg_{c}") for c in cols_to_sum]

result = df.agg(*agg_exprs)

print("=== 패턴 6: 여러 컬럼 일괄 집계 ===")
result.show()

### 실습

#### 실습 5-1: 조건부 집계

부서별로 전체 인원과 급여 80000 이상인 인원 수를 집계하세요.

<details>
<summary>힌트</summary>

`count(when(조건, 1))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.groupBy("department").agg(
    count("*").alias("total"),
    count(when(col("salary") >= 80000, 1)).alias("high_salary_cnt")
).show()
```

</details>

#### 실습 5-2: 그룹 내 비율

Window 함수를 사용하여 각 직원의 급여가 부서 전체 급여에서 차지하는 비율(%)을 계산하세요.

<details>
<summary>힌트</summary>

`sum("salary").over(Window.partitionBy("department"))`

</details>

<details>
<summary>모범 답안</summary>

```python
window_dept = Window.partitionBy("department")
df.filter(col("salary").isNotNull()).withColumn(
    "dept_total", sum("salary").over(window_dept)
).withColumn(
    "pct", spark_round(col("salary") / col("dept_total") * 100, 2)
).select("department", "name", "salary", "pct").show(10)
```

</details>

#### 실습 5-3: 데이터 품질 체크

전체 행 수, salary NULL 개수, department NULL 개수를 한 번에 조회하세요.

<details>
<summary>힌트</summary>

`count(when(col("컬럼").isNull(), 1))`

</details>

<details>
<summary>모범 답안</summary>

```python
df.agg(
    count("*").alias("total_rows"),
    count(when(col("salary").isNull(), 1)).alias("salary_null"),
    count(when(col("department").isNull(), 1)).alias("dept_null")
).show()
```

</details>

#### 실습 5-4: 컬럼명 일괄 변경

DataFrame의 모든 컬럼명을 소문자로 변경하세요.

<details>
<summary>힌트</summary>

`df.toDF(*[c.lower() for c in df.columns])`

</details>

<details>
<summary>모범 답안</summary>

```python
new_cols = [c.lower() for c in df.columns]
df.toDF(*new_cols).printSchema()
```

</details>

---

## 퀴즈

Q1. fillna({"salary": 0, "department": "Unknown"})의 동작은?

- A) 모든 컬럼의 NULL을 0으로 채운다
- B) salary의 NULL은 0, department의 NULL은 Unknown으로 채운다
- C) salary와 department가 모두 NULL인 행만 채운다
- D) 에러가 발생한다

<details>
<summary>정답 보기</summary>

**정답: B) salary의 NULL은 0, department의 NULL은 Unknown으로 채운다**

딕셔너리로 컬럼별 다른 값을 지정할 수 있습니다.

</details>

---

Q2. row_number()와 rank()의 차이점은?

- A) row_number는 정렬 필요, rank는 정렬 불필요
- B) row_number는 동점도 순차 번호, rank는 동점이면 같은 순위
- C) row_number는 그룹별, rank는 전체
- D) 차이 없음

<details>
<summary>정답 보기</summary>

**정답: B) row_number는 동점도 순차 번호, rank는 동점이면 같은 순위**

동점 데이터 예시:
- row_number: 1, 2, 3, 4
- rank: 1, 1, 3, 4 (동점은 같은 순위, 다음 건너뜀)
- dense_rank: 1, 1, 2, 3 (동점은 같은 순위, 순위 안 건너뜀)

</details>

---

Q3. 7일 이동 평균을 구하려면 rowsBetween의 값은?

- A) `rowsBetween(0, 6)`
- B) `rowsBetween(-6, 0)`
- C) `rowsBetween(-7, -1)`
- D) `rowsBetween(1, 7)`

<details>
<summary>정답 보기</summary>

**정답: B) `rowsBetween(-6, 0)`**

-6은 현재 행 기준 6행 전, 0은 현재 행입니다.
현재 행 포함 최근 7일 = 현재행 + 이전 6행 = -6 ~ 0

</details>

---

Q4. coalesce(col1, col2, lit("default"))의 동작은?

- A) col1이 NULL이면 col2, col2도 NULL이면 "default"
- B) col1과 col2를 합친다
- C) col1, col2, "default" 중 가장 큰 값 반환
- D) col1, col2의 NULL 개수 반환

<details>
<summary>정답 보기</summary>

**정답: A) col1이 NULL이면 col2, col2도 NULL이면 "default"**

coalesce는 왼쪽부터 확인하여 첫 번째 non-null 값을 반환합니다.

</details>

---

## 과제: 직원 성과 분석 ETL 파이프라인

### 시나리오

데이터 엔지니어로서 **직원 성과 분석 시스템**을 구축해야 합니다.
원본 데이터는 품질 이슈(NULL, 공백, 형식 불일치)가 있어 정제가 필요하고,
날짜 기반 파생 변수와 Window 함수를 활용한 성과 지표 계산이 요구됩니다.

**파이프라인 단계:**
1. 데이터 클렌징 (NULL 처리, 문자열 정리)
2. 날짜 파생 변수 생성
3. Window 함수로 성과 지표 계산
4. 데이터 품질 검증
5. 최종 분석용 데이터셋 생성

---

### Step 1: 데이터 클렌징

원본 직원 데이터(`df`)의 품질 이슈를 해결하세요:

1. `name` 컬럼의 앞뒤 공백 제거
2. `salary`가 NULL인 경우 부서 평균으로 채우기 (간단히 60000으로 대체)
3. `department`가 NULL인 경우 "Unassigned"로 채우기
4. `email`에서 사용자명(@ 앞부분)만 추출한 `username` 컬럼 추가

<details>
<summary>힌트</summary>

- `trim(col("name"))` - 공백 제거
- `fillna({"컬럼": 값})` - NULL 대체
- `split(col("email"), "@")[0]` - @ 기준 분리

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
df_step1 = (
    df
    .withColumn("name", trim(col("name")))
    .fillna({"salary": 60000, "department": "Unassigned"})
    .withColumn("username", split(col("email"), "@")[0])
)

print("=== Step 1: 데이터 클렌징 완료 ===")
df_step1.select("name", "email", "username", "salary", "department").show(5)

# NULL 개수 확인
print("NULL 개수 확인:")
df_step1.select(
    count(when(col("salary").isNull(), 1)).alias("salary_null"),
    count(when(col("department").isNull(), 1)).alias("dept_null")
).show()

</details>

---

### Step 2: 날짜 파생 변수 생성

`join_date`를 파싱하여 다음 파생 컬럼을 추가하세요:

1. `join_date_dt`: 문자열을 날짜 타입으로 변환
2. `입사연도`: 입사 연도
3. `입사월`: 입사 월
4. `근속일수`: 오늘까지의 근속 일수
5. `근속년수`: 근속일수 / 365 (소수점 1자리)
6. `입사분기`: 입사 월 기준 분기 (1~3월: Q1, 4~6월: Q2, ...)

<details>
<summary>힌트</summary>

- `to_date(col, "yyyy-MM-dd")` - 날짜 파싱
- `year()`, `month()` - 연/월 추출
- `datediff(current_date(), col)` - 날짜 차이
- `when()` 체이닝으로 분기 계산

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
df_step2 = (
    df_step1
    .withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd"))
    .withColumn("입사연도", year(col("join_date_dt")))
    .withColumn("입사월", month(col("join_date_dt")))
    .withColumn("근속일수", datediff(current_date(), col("join_date_dt")))
    .withColumn("근속년수", spark_round(col("근속일수") / 365, 1))
    .withColumn(
        "입사분기",
        when(col("입사월") <= 3, "Q1")
        .when(col("입사월") <= 6, "Q2")
        .when(col("입사월") <= 9, "Q3")
        .otherwise("Q4")
    )
)

print("=== Step 2: 날짜 파생 변수 ===")
df_step2.select(
    "name", "join_date", "입사연도", "입사월", "입사분기", "근속일수", "근속년수"
).show(10)

</details>

---

### Step 3: Window 함수로 성과 지표 계산

Window 함수를 활용하여 다음 성과 지표를 계산하세요:

1. `부서내순위`: 부서별 급여 내림차순 순위 (row_number)
2. `부서평균급여`: 해당 직원이 속한 부서의 평균 급여
3. `급여편차`: 본인 급여 - 부서 평균 급여
4. `부서내비율`: 본인 급여 / 부서 총급여 × 100 (%)

<details>
<summary>힌트</summary>

- `Window.partitionBy("department")` - 부서별 그룹
- `row_number().over(window)` - 순위
- `avg().over(window)` - 그룹 평균
- `sum().over(window)` - 그룹 합계

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
from pyspark.sql.window import Window

# Window 정의
window_dept = Window.partitionBy("department")
window_dept_rank = Window.partitionBy("department").orderBy(col("salary").desc())

df_step3 = (
    df_step2
    .withColumn("부서내순위", row_number().over(window_dept_rank))
    .withColumn("부서평균급여", spark_round(avg("salary").over(window_dept), 0))
    .withColumn("급여편차", col("salary") - col("부서평균급여"))
    .withColumn(
        "부서내비율",
        spark_round(col("salary") / sum("salary").over(window_dept) * 100, 1)
    )
)

print("=== Step 3: 성과 지표 ===")
df_step3.select(
    "department", "name", "salary", "부서내순위", "부서평균급여", "급여편차", "부서내비율"
).orderBy("department", "부서내순위").show(15)

</details>

---

### Step 4: 데이터 품질 검증

최종 데이터의 품질을 검증하는 리포트를 생성하세요:

1. 전체 행 수
2. 각 컬럼별 NULL 개수
3. 급여 통계 (최소, 최대, 평균)
4. 부서별 인원 분포
5. 중복 emp_id 존재 여부

<details>
<summary>힌트</summary>

- `count(when(col.isNull(), 1))` - NULL 카운트
- `min()`, `max()`, `avg()` - 통계
- `groupBy().count()` - 분포
- `groupBy("emp_id").count().filter(col("count") > 1)` - 중복 확인

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
print("=== Step 4: 데이터 품질 검증 ===")

# 1. 전체 행 수
print(f"전체 행 수: {df_step3.count()}")

# 2. NULL 개수
print("\n[NULL 개수]")
df_step3.select(
    count(when(col("salary").isNull(), 1)).alias("salary_null"),
    count(when(col("department").isNull(), 1)).alias("dept_null"),
    count(when(col("join_date_dt").isNull(), 1)).alias("date_null")
).show()

# 3. 급여 통계
print("[급여 통계]")
df_step3.agg(
    min("salary").alias("최소급여"),
    max("salary").alias("최대급여"),
    spark_round(avg("salary"), 0).alias("평균급여")
).show()

# 4. 부서별 인원 분포
print("[부서별 인원 분포]")
df_step3.groupBy("department").count().orderBy(col("count").desc()).show()

# 5. 중복 emp_id 확인
dup_count = df_step3.groupBy("emp_id").count().filter(col("count") > 1).count()
print(f"[중복 emp_id 개수]: {dup_count}")

</details>

---

### Step 5: 최종 분석용 데이터셋 생성

최종 분석용 데이터셋을 생성하세요:

1. 부서가 "Unassigned"가 아닌 직원만 포함
2. 부서별 급여 상위 3명만 추출
3. 필요한 컬럼만 선택: `emp_id`, `name`, `department`, `salary`, `근속년수`, `부서내순위`, `급여편차`
4. 부서명, 급여 순으로 정렬

<details>
<summary>힌트</summary>

- `filter(col("department") != "값")` - 필터링
- `filter(col("부서내순위") <= 3)` - Top N
- `select()` - 컬럼 선택
- `orderBy()` - 정렬

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
df_final = (
    df_step3
    .filter(col("department") != "Unassigned")
    .filter(col("부서내순위") <= 3)
    .select(
        "emp_id", "name", "department", "salary",
        "근속년수", "부서내순위", "급여편차"
    )
    .orderBy("department", col("salary").desc())
)

print("=== 최종 분석용 데이터셋 ===")
df_final.show()
print(f"최종 데이터 건수: {df_final.count()}")

</details>

---

### 최종 과제: 전체 ETL 파이프라인 완성

위 Step 1~5를 하나의 완전한 ETL 파이프라인으로 작성하세요.
각 단계를 함수로 분리하여 모듈화하면 보너스 점수!

<details>
<summary>힌트</summary>

- 각 단계를 함수로 정의
- 함수들을 순차 호출하여 파이프라인 구성
- 최종 결과 반환

</details>

<details>
<summary>모범 답안</summary>

In [ ]:
from pyspark.sql.window import Window

def clean_data(df):
    """Step 1: 데이터 클렌징"""
    return (
        df
        .withColumn("name", trim(col("name")))
        .fillna({"salary": 60000, "department": "Unassigned"})
        .withColumn("username", split(col("email"), "@")[0])
    )

def add_date_features(df):
    """Step 2: 날짜 파생 변수"""
    return (
        df
        .withColumn("join_date_dt", to_date(col("join_date"), "yyyy-MM-dd"))
        .withColumn("입사연도", year(col("join_date_dt")))
        .withColumn("근속일수", datediff(current_date(), col("join_date_dt")))
        .withColumn("근속년수", spark_round(col("근속일수") / 365, 1))
    )

def add_performance_metrics(df):
    """Step 3: 성과 지표 계산"""
    window_dept = Window.partitionBy("department")
    window_rank = Window.partitionBy("department").orderBy(col("salary").desc())

    return (
        df
        .withColumn("부서내순위", row_number().over(window_rank))
        .withColumn("부서평균급여", spark_round(avg("salary").over(window_dept), 0))
        .withColumn("급여편차", col("salary") - col("부서평균급여"))
    )

def create_final_dataset(df):
    """Step 5: 최종 데이터셋"""
    return (
        df
        .filter(col("department") != "Unassigned")
        .filter(col("부서내순위") <= 3)
        .select(
            "emp_id", "name", "department", "salary",
            "근속년수", "부서내순위", "급여편차"
        )
        .orderBy("department", col("salary").desc())
    )

# 전체 파이프라인 실행
print("=== ETL 파이프라인 실행 ===")

df_result = (
    df
    .transform(clean_data)
    .transform(add_date_features)
    .transform(add_performance_metrics)
    .transform(create_final_dataset)
)

print("=== 최종 결과 ===")
df_result.show()
print(f"처리 완료: {df_result.count()}건")

</details>

---

## 핵심 요약

### NULL 처리

```python
df.dropna()                          # NULL 행 제거
df.dropna(subset=["col"])            # 특정 컬럼 NULL 제거
df.fillna(0)                         # NULL → 0
df.fillna({"col1": 0, "col2": "X"})  # 컬럼별 다른 값
coalesce(col1, col2, lit("기본값"))   # 첫 번째 non-null
```

### 문자열 함수

```python
trim(col)                            # 공백 제거
upper(col) / lower(col)              # 대소문자
concat_ws("_", col1, col2)           # 구분자로 합치기
split(col, "@")[0]                   # 분리 후 인덱싱
regexp_replace(col, 패턴, 대체)       # 정규식 치환
```

### 날짜 함수

```python
to_date(col, "yyyy-MM-dd")           # 문자열 → 날짜
date_format(col, "yyyy/MM")          # 포맷 변경
year(col), month(col)                # 부분 추출
datediff(end, start)                 # 날짜 차이
date_add(col, 7)                     # 날짜 더하기
```

### Window 함수

```python
window = Window.partitionBy("그룹").orderBy("정렬")

row_number().over(window)            # 순위 (1,2,3,4)
rank().over(window)                  # 순위 (동점 같은 순위)
lag("col", 1).over(window)           # 이전 행
lead("col", 1).over(window)          # 다음 행
sum("col").over(window_cumsum)       # 누적합
avg("col").over(window_ma7)          # 이동평균
```

In [ ]:
# 세션 종료
# spark.stop()
print("\n4교시 완료! PySpark 기초 API 학습 완료!")
print("다음 단계: Spark Structured Streaming")

# Day 12 - 4교시: Spark Structured Streaming

![Spark Structured Streaming](https://moons08.github.io/assets/img/post/spark/streaming-arch.png)

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- Spark Structured Streaming의 개념과 특징을 이해할 수 있다
- Kafka에서 실시간으로 데이터를 읽어올 수 있다
- 윈도우 기반 집계(시간 윈도우)를 구현할 수 있다
- 스트리밍 결과를 콘솔/파일/Kafka로 출력할 수 있다

---

## 핵심 개념 1: 배치 처리 vs 스트리밍 처리

### 두 가지 데이터 처리 방식

데이터 처리 방식은 크게 **배치(Batch)**와 **스트리밍(Streaming)**으로 나뉩니다.

![](https://k21academy.com/wp-content/uploads/2020/11/BatchProcessingStreamProcessing_Diagram-02.png)

### 배치 처리 (Batch Processing)

**흐름**: 어제의 데이터 → 처리 (한번에) → 결과 (다음날 확인)

**예시**:
- 매일 밤 12시에 전날 판매 리포트 생성
- 매주 월요일에 주간 사용자 통계 계산
- 매월 1일에 월간 정산 처리

**특징**:
- 데이터가 모두 준비된 후 처리
- 높은 처리량 (throughput) 가능
- 지연 시간(latency)이 김 (시간~일)

### 스트리밍 처리 (Stream Processing)

**흐름**: 실시간 데이터 → 처리 → 결과 → 처리 → 결과 (끊임없이 반복)

**예시**:
- 실시간 대시보드 (트래픽 모니터링)
- 이상 거래 탐지 (1초 내 알림)
- 실시간 추천 시스템

**특징**:
- 데이터가 도착하는 즉시 처리
- 낮은 지연 시간(latency) (밀리초~초)
- 24시간 계속 실행

### 비교 요약

| 구분 | 배치 처리 | 스트리밍 처리 |
|------|----------|--------------|
| 처리 시점 | 데이터 축적 후 일괄 처리 | 데이터 도착 즉시 처리 |
| 지연 시간 | 시간 ~ 일 | 밀리초 ~ 초 |
| 처리량 | 높음 | 상대적으로 낮음 |
| 실행 방식 | 스케줄 기반 (주기적) | 24시간 상시 실행 |
| 사용 사례 | 리포트, 정산, 분석 | 모니터링, 알림, 실시간 대시보드 |

### 언제 어떤 방식을 사용하나요?

| 상황 | 배치 | 스트리밍 |
|------|------|----------|
| 데이터 분석 리포트 | O | |
| 실시간 대시보드 | | O |
| 월간 정산 | O | |
| 이상 거래 탐지 | | O |
| 머신러닝 모델 학습 | O | |
| 실시간 추천 | | O |
| ETL 파이프라인 | O | O (둘 다 가능) |

## 핵심 개념 2: Micro-batch 처리 방식

### Spark Structured Streaming의 동작 원리

Spark Structured Streaming은 **Micro-batch** 방식으로 스트리밍을 처리합니다.
데이터를 작은 배치 단위로 나누어 처리하는 방식입니다.

**처리 흐름**:
1. 데이터 도착 → 배치 단위로 분할
2. 각 배치별 처리 실행
3. 결과 출력
4. 다음 배치 반복

예: `trigger: processingTime="5 seconds"` → 5초마다 배치 처리

### Micro-batch의 장점

| 장점 | 설명 |
|------|------|
| **Exactly-once 보장** | 데이터 중복/손실 없이 정확히 한 번 처리 |
| **배치 코드 재사용** | DataFrame API를 그대로 사용 |
| **장애 복구 용이** | 체크포인트로 상태 저장/복구 |
| **간편한 개발** | 스트리밍 복잡성을 추상화 |

### 트리거(Trigger) 옵션

| 트리거 | 설명 | 사용 예시 |
|--------|------|----------|
| `processingTime="5 seconds"` | 5초마다 배치 처리 | 실시간 대시보드 |
| `processingTime="1 minute"` | 1분마다 배치 처리 | 준실시간 집계 |
| `once=True` | 한 번만 처리 | 배치 스타일 스트리밍 |
| `availableNow=True` | 현재까지 데이터 한 번 처리 | 백필(backfill) |

### 리얼타임 모드(Real-Time Mode, RTM)

Spark 4.1부터 도입된 **초저지연 연속 처리** 모드입니다.
Micro-batch의 배치 간격으로 인한 지연을 제거하여 **p99 레이턴시를 한 자릿수 밀리초**까지 낮출 수 있습니다.

**Micro-batch vs Real-Time Mode**:

![Code Diff btw rtm simple change](https://www.databricks.com/sites/default/files/inline-images/image5_36.png)

| 구분 | Micro-batch | Real-Time Mode |
|------|-------------|----------------|
| **처리 방식** | 주기적 배치 처리 | 연속(Continuous) 처리 |
| **레이턴시** | 초~분 단위 | 밀리초 단위 |
| **적용 방식** | 기본 모드 | 설정 변경만으로 활성화 |

**RTM 지원 범위 (Spark 4.1 기준)**:

| 항목 | 지원 내용 |
|------|-----------|
| **언어** | Scala |
| **쿼리 타입** | Stateless / 단일 스테이지 |
| **Source** | Kafka |
| **Sink** | Kafka, ForeachSink |
| **출력 모드** | Update 모드만 |

> 💡 **선택 기준**: 대부분의 스트리밍 워크로드는 Micro-batch로 충분합니다.
> 밀리초 단위 응답이 필수인 경우에만 RTM을 고려하세요.

## 핵심 개념 3: Watermark (워터마크)

### 지연 데이터 문제

실시간 시스템에서는 네트워크 지연 등으로 데이터가 늦게 도착할 수 있습니다.
Watermark는 "얼마나 늦은 데이터까지 허용할 것인가"를 정의합니다.

**예시**: Watermark = 10분 설정

| 실제 시간 | 데이터 | 이벤트 시간 | 처리 여부 |
|-----------|--------|-------------|-----------|
| 10:00 | A | 10:00 | 처리 |
| 10:05 | B | 10:04 | 처리 |
| 10:10 | C | 10:08 | 처리 |
| 10:15 | D | 10:12 | 처리 |
| 10:20 | E | 10:03 (늦음!) | **버림** |

**계산**: 현재 시간 10:15일 때, Watermark = 10:15 - 10분 = 10:05
- 이벤트 시간 10:03인 (E)는 10:05보다 이전 → **버림**
- 결론: 10분 이상 늦은 데이터는 무시됨

### Watermark 설정 방법

```python
df.withWatermark("event_time", "10 minutes")
```

| 파라미터 | 의미 |
|----------|------|
| `"event_time"` | 이벤트 시간 컬럼명 |
| `"10 minutes"` | 허용할 최대 지연 시간 |

### Watermark 설정 기준

| 값 | 상황 |
|----|------|
| 작은 값 (1분) | 네트워크가 안정적, 빠른 결과 필요 |
| 큰 값 (1시간) | 네트워크가 불안정, 데이터 손실 최소화 |
| 중간 값 (10분) | 일반적인 상황 |

## 핵심 개념 4: 윈도우 집계 (Window Aggregation)

### 시간 윈도우란?

![](https://www.databricks.com/wp-content/uploads/2021/10/Native-Support-4-Session-Window-in-Spark-Streaming-blog-img-1.jpg)

스트리밍 데이터를 시간 단위로 묶어서 집계하는 방법입니다.

### Tumbling Window (텀블링 윈도우)

윈도우가 **겹치지 않음** - 각 데이터는 하나의 윈도우에만 속함

| 윈도우 | 시간 범위 | 설명 |
|--------|----------|------|
| Window 1 | 0~5분 | 집계 결과 1 |
| Window 2 | 5~10분 | 집계 결과 2 |
| Window 3 | 10~15분 | 집계 결과 3 |

**특징**: 가장 일반적으로 사용

### Sliding Window (슬라이딩 윈도우)

윈도우가 **겹침** - 하나의 데이터가 여러 윈도우에 속할 수 있음

| 윈도우 | 시간 범위 |
|--------|----------|
| Window 1 | 0~5분 |
| Window 2 | 2~7분 |
| Window 3 | 4~9분 |
| Window 4 | 6~11분 |

**특징**: 더 세밀한 분석 가능

### 윈도우 설정 방법

```python
# Tumbling Window (5분)
window(col("event_time"), "5 minutes")

# Sliding Window (5분 윈도우, 1분 슬라이드)
window(col("event_time"), "5 minutes", "1 minute")
```

## Structured Streaming이란?

### 개념

**배치 처리 (기존 방식)**: 데이터 전체 로드 → 처리 → 결과 출력

**스트리밍 처리 (Structured Streaming)**: 데이터 조금씩 → 처리 → 결과 지속 업데이트 (micro-batch 반복)

### 핵심 아이디어: 무한 테이블

스트리밍 데이터를 "무한히 늘어나는 테이블"처럼 취급합니다.

| 시점 | 테이블 상태 |
|------|------------|
| 초기 | row 1, row 2 |
| 새 데이터 도착 | row 1, row 2, **row 3 (new)**, **row 4 (new)** |
| 또 새 데이터 | row 1, row 2, row 3, row 4, **row 5 (new)** |

**장점**: 같은 DataFrame API로 배치/스트리밍 모두 처리 가능

---

## Part 1: Kafka → Spark 연결

### Kafka 스트림 읽기

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
)

# =============================================================================
# 1. SparkSession 생성
# =============================================================================
# Kafka 연동을 위해 spark-sql-kafka 패키지가 필요합니다.
# 패키지 버전은 Spark 버전과 일치해야 합니다 (Spark 3.5.8 → 패키지 3.5.8)

spark = (
    SparkSession.builder
    .appName("Day11-Streaming")
    .master("spark://spark-master:7077")

    # Kafka 연동 패키지: org.apache.spark:spark-sql-kafka-0-10_2.12:<spark버전>
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")

    # 체크포인트: 스트리밍 상태를 저장하여 장애 복구 시 사용
    .config("spark.sql.streaming.checkpointLocation", "/data/checkpoints")
    .getOrCreate()
)

print("SparkSession 생성 완료!")
print(f"  App ID: {spark.sparkContext.applicationId}")

# =============================================================================
# 2. 스키마 정의
# =============================================================================
# Kafka 메시지는 바이너리(binary)로 전송되므로, JSON 파싱을 위해 스키마가 필요합니다.
# StructType: 여러 필드로 구성된 구조체 타입
# StructField(이름, 타입, nullable): 각 필드 정의

schema = StructType([
    StructField("request_id", StringType(), True),       # 요청 고유 ID
    StructField("user_id", StringType(), True),          # 사용자 ID
    StructField("endpoint", StringType(), True),         # API 엔드포인트 (/api/users 등)
    StructField("method", StringType(), True),           # HTTP 메서드 (GET, POST 등)
    StructField("status_code", IntegerType(), True),     # HTTP 상태 코드 (200, 404, 500 등)
    StructField("response_time_ms", IntegerType(), True),# 응답 시간 (밀리초)
    StructField("timestamp", StringType(), True),        # 이벤트 발생 시간
])

print("\n스키마 정의:")
print(schema)

# =============================================================================
# 3. Kafka에서 스트림 읽기
# =============================================================================
# readStream: 스트리밍 데이터를 읽는 메서드 (배치는 read)
# format("kafka"): Kafka를 데이터 소스로 사용
# kafka.bootstrap.servers: Kafka 브로커 주소
# subscribe: 구독할 토픽 이름
# startingOffsets: 어디서부터 읽을지 (latest: 최신, earliest: 처음부터)

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")  # Kafka 브로커 주소
    .option("subscribe", "api-events")                 # 구독할 토픽
    .option("startingOffsets", "latest")               # 최신 메시지부터 읽기
    .load()
)

print("\nKafka 스트림 연결 완료!")
print(f"  소스 스키마: {df_raw.schema.simpleString()}")

# =============================================================================
# 4. JSON 파싱 및 타입 변환
# =============================================================================
# Kafka 메시지 구조: key(binary), value(binary), topic, partition, offset, timestamp
# value 컬럼에 실제 JSON 데이터가 들어있음
#
# 처리 과정:
# 1) col("value").cast("string"): 바이너리 → 문자열 변환
# 2) from_json(..., schema): JSON 문자열 → 구조체로 파싱
# 3) .alias("data"): 파싱된 결과에 별칭 부여
# 4) .select("data.*"): 구조체 내부 필드들을 개별 컬럼으로 펼침
# 5) to_timestamp(): 문자열 → 타임스탬프 타입 변환

df_parsed = (
    df_raw
    .select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")  # data.request_id, data.user_id, ... → request_id, user_id, ...
    .withColumn("event_time", to_timestamp(col("timestamp")))  # 윈도우 집계용 타임스탬프
)

print("\n파싱된 스키마:")
df_parsed.printSchema()

**예상 출력**:

```
SparkSession 생성 완료!
  App ID: app-20260118-...

스키마 정의:
StructType([StructField('request_id', StringType(), True), ...])

Kafka 스트림 연결 완료!
  소스 스키마: struct<key:binary,value:binary,topic:string,...>

파싱된 스키마:
root
 |-- request_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
```

---

## Part 2: 콘솔 출력 스트리밍

### writeStream으로 스트림 출력하기

스트리밍 데이터를 처리한 후 결과를 출력하려면 `writeStream`을 사용합니다.

In [ ]:
# =============================================================================
# 스트림 출력 (writeStream)
# =============================================================================
# writeStream: 스트리밍 결과를 출력하는 메서드
# outputMode: 출력 방식 지정
# format: 출력 대상 (console, parquet, kafka 등)
# trigger: 처리 주기 설정
# start(): 스트리밍 쿼리 시작

query = (
    df_parsed.writeStream

    # outputMode: 어떤 데이터를 출력할지 결정
    # - "append": 새로 추가된 행만 출력 (집계 없는 단순 처리에 사용)
    # - "complete": 전체 결과 출력 (집계 결과 전체를 매번 출력)
    # - "update": 변경된 행만 출력 (집계 결과 중 업데이트된 것만)
    .outputMode("append")

    # format: 출력 대상 지정
    # - "console": 터미널에 출력 (개발/디버깅용)
    # - "parquet": Parquet 파일로 저장
    # - "kafka": 다른 Kafka 토픽으로 전송
    .format("console")

    # truncate=False: 긴 문자열을 자르지 않고 전체 출력
    .option("truncate", False)

    # trigger: 얼마나 자주 배치를 처리할지
    # - processingTime="5 seconds": 5초마다 배치 처리
    # - once=True: 한 번만 처리 후 종료
    # - availableNow=True: 현재까지 도착한 데이터만 처리
    .trigger(processingTime="5 seconds")

    # start(): 스트리밍 쿼리 시작 (비동기 실행)
    .start()
)

print("스트리밍 시작! (Ctrl+C로 종료)")

# awaitTermination(): 쿼리가 종료될 때까지 대기
# 이 줄이 없으면 프로그램이 바로 종료됨
query.awaitTermination()

### writeStream 옵션 요약

| 옵션 | 설명 | 값 예시 |
|------|------|---------|
| `outputMode` | 출력 모드 | `append`, `complete`, `update` |
| `format` | 출력 형식 | `console`, `parquet`, `kafka` |
| `trigger` | 실행 주기 | `processingTime="5 seconds"` |
| `checkpointLocation` | 체크포인트 저장 위치 | `/data/checkpoints` |

### outputMode 비교

| 모드 | 설명 | 사용 상황 |
|------|------|----------|
| `append` | 새 행만 출력 | 집계 없는 단순 처리 |
| `complete` | 전체 결과 출력 | 집계 결과 전체 출력 |
| `update` | 변경된 행만 출력 | 집계 결과 중 변경분만 |

---

## Part 3: 윈도우 집계

### 시간 윈도우 집계란?

스트리밍 데이터를 **시간 단위로 묶어서 집계**하는 방법입니다.
예를 들어 "5분마다 요청 수, 평균 응답시간, 에러율 계산"이 가능합니다.

### 윈도우 집계 코드

In [ ]:
from pyspark.sql.functions import col, window, count, avg, sum as spark_sum, when

# =============================================================================
# 윈도우 집계
# =============================================================================
# 스트리밍 데이터를 시간 기준으로 그룹화하여 집계합니다.

df_windowed = (
    df_parsed

    # -------------------------------------------------------------------------
    # withWatermark: 지연 데이터 허용 범위 설정
    # -------------------------------------------------------------------------
    # 네트워크 지연 등으로 늦게 도착하는 데이터를 얼마나 기다릴지 설정
    # "10 minutes": 이벤트 시간 기준 10분까지 늦은 데이터는 처리
    # 10분 이상 늦은 데이터는 무시됨 (메모리 관리를 위해 필요)
    .withWatermark("event_time", "10 minutes")

    # -------------------------------------------------------------------------
    # groupBy + window: 시간 윈도우로 그룹화
    # -------------------------------------------------------------------------
    # window(시간컬럼, 윈도우크기): 지정된 크기의 시간 윈도우로 데이터 그룹화
    # - "5 minutes": 5분 단위로 데이터를 묶음
    # - 0~5분, 5~10분, 10~15분... 각각 별도 그룹
    .groupBy(
        window(col("event_time"), "5 minutes"),  # 5분 윈도우
        col("endpoint")                           # 엔드포인트별로도 그룹화
    )

    # -------------------------------------------------------------------------
    # agg: 집계 함수 적용
    # -------------------------------------------------------------------------
    .agg(
        # count(): 행 개수 세기
        count("request_id").alias("request_count"),

        # avg(): 평균 계산
        avg("response_time_ms").alias("avg_response_time"),

        # 조건부 합계: status_code >= 400이면 1, 아니면 0을 더함
        # when(조건, 참일때값).otherwise(거짓일때값)
        spark_sum(
            when(col("status_code") >= 400, 1).otherwise(0)
        ).alias("error_count"),
    )

    # 에러율 계산: 에러 수 / 전체 요청 수 * 100
    .withColumn("error_rate", col("error_count") / col("request_count") * 100)
)

# =============================================================================
# 윈도우 집계 결과 출력
# =============================================================================
# 집계 결과는 outputMode="update" 사용
# - append: 집계에서는 사용 불가 (윈도우가 닫히기 전까지 결과가 변함)
# - complete: 전체 결과 매번 출력 (데이터가 많으면 비효율)
# - update: 변경된 윈도우만 출력 (권장)

query = (
    df_windowed.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)

print("5분 윈도우 집계 시작!")
query.awaitTermination()

**예상 출력** (Producer가 메시지 전송 중일 때):

```
-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
|window                                    |endpoint       |request_count|avg_response_time|error_count|error_rate|
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/products  |          156|            245.3|         39|      25.0|
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/users     |          142|            267.1|         35|      24.6|
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/orders    |          138|            251.8|         33|      23.9|
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
```

---

## 실습 과제

5교시에서 배운 내용을 종합적으로 연습합니다.

| 과제 | 연습 개념 |
|------|----------|
| 과제 1 | 윈도우 크기 변경, trigger 설정 |
| 과제 2 | outputMode, format 변경 (Parquet 저장) |
| 과제 3 | filter + 윈도우 집계 조합 |
| 과제 4 | Watermark 조정 |
| 보너스 | Kafka Sink (to_json, struct) |

---

### 과제 1: 윈도우 크기와 Trigger 변경

**목표**: 5분 윈도우를 **1분 윈도우**로 변경하고, trigger를 **10초**로 설정하세요.

<details>
<summary>힌트 보기</summary>

- `window()` 함수의 두 번째 인자가 윈도우 크기입니다
- `trigger(processingTime=...)` 으로 배치 처리 주기를 설정합니다

</details>

<details>
<summary>모범 답안</summary>

```python
df_windowed = (
    df_parsed
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),  # 5 minutes → 1 minute
        col("endpoint")
    )
    .agg(
        count("request_id").alias("request_count"),
        avg("response_time_ms").alias("avg_response_time"),
    )
)

query = (
    df_windowed.writeStream
    .outputMode("update")
    .format("console")
    .trigger(processingTime="10 seconds")  # 10초마다 배치 처리
    .start()
)
```

</details>

---

### 과제 2: Parquet 파일 저장

**목표**: 집계 결과를 **Parquet 파일**로 저장하세요.

**조건**:
- 저장 경로: `/data/output/api_stats`
- 체크포인트: `/data/checkpoints/parquet`
- 1분마다 저장

<details>
<summary>힌트 보기</summary>

- `format("parquet")`으로 Parquet 형식 지정
- `option("path", ...)`로 저장 경로 지정
- 파일 저장 시에는 `outputMode("append")` 사용 (윈도우가 닫힌 후 저장)
- `checkpointLocation` 옵션 필수!

</details>

<details>
<summary>모범 답안</summary>

```python
query = (
    df_windowed.writeStream
    .outputMode("append")           # 파일 저장은 append 모드
    .format("parquet")              # Parquet 형식으로 저장
    .option("path", "/data/output/api_stats")
    .option("checkpointLocation", "/data/checkpoints/parquet")
    .trigger(processingTime="1 minute")
    .start()
)

print("Parquet 저장 시작!")
print("  저장 경로: /data/output/api_stats")
query.awaitTermination()
```

**참고**: `append` 모드에서는 윈도우가 완전히 닫힌 후에만 결과가 출력됩니다.
Watermark 시간이 지나야 윈도우가 닫히므로, 결과가 나오기까지 시간이 걸릴 수 있습니다.

</details>

---

### 과제 3: 에러 필터링 집계

**목표**: status_code가 400 이상인 **에러만 필터링**하여 1분 윈도우로 집계하세요.

**출력 컬럼**: endpoint, status_code, error_count

<details>
<summary>힌트 보기</summary>

- `filter(col("status_code") >= 400)`로 에러만 선택
- 필터링 후 `groupBy`로 집계
- status_code별로도 그룹화하면 어떤 에러가 많은지 파악 가능

</details>

<details>
<summary>모범 답안</summary>

```python
# 1. 에러만 필터링
df_errors = df_parsed.filter(col("status_code") >= 400)

# 2. 1분 윈도우 에러 집계
df_error_stats = (
    df_errors
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("endpoint"),
        col("status_code")
    )
    .agg(count("request_id").alias("error_count"))
)

# 3. 콘솔 출력
query = (
    df_error_stats.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)

print("에러 모니터링 시작!")
print("  - status_code >= 400 필터링")
print("  - 1분 윈도우 집계")
query.awaitTermination()
```

</details>

---

### 과제 4: Watermark 조정

**목표**: 네트워크가 불안정한 환경을 가정하고, **Watermark를 30분**으로 늘려보세요.

**질문**: Watermark를 늘리면 어떤 장단점이 있을까요?

<details>
<summary>힌트 보기</summary>

- `withWatermark("event_time", "30 minutes")`
- Watermark가 길수록:
  - 장점: 늦게 도착하는 데이터도 처리 가능
  - 단점: ???

</details>

<details>
<summary>모범 답안</summary>

```python
df_windowed = (
    df_parsed
    .withWatermark("event_time", "30 minutes")  # 10분 → 30분
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("endpoint")
    )
    .agg(
        count("request_id").alias("request_count"),
        avg("response_time_ms").alias("avg_response_time"),
    )
)
```

**Watermark 조정의 트레이드오프**:

| Watermark | 장점 | 단점 |
|-----------|------|------|
| **짧게 (1분)** | 빠른 결과 출력, 메모리 적게 사용 | 늦은 데이터 손실 |
| **길게 (30분)** | 늦은 데이터도 처리 | 결과 출력 지연, 메모리 많이 사용 |

**선택 기준**:
- 네트워크 안정적 → 짧은 Watermark (1~5분)
- 네트워크 불안정 → 긴 Watermark (10~30분)
- 데이터 정확성 중요 → 긴 Watermark
- 빠른 응답 중요 → 짧은 Watermark

</details>

---

## 보너스: Kafka로 결과 전송

**목표**: 집계 결과를 다른 Kafka 토픽(`api-events-aggregated`)으로 전송하세요.

Kafka는 key-value 형태로 메시지를 전송하므로, DataFrame을 JSON으로 변환해야 합니다.

<details>
<summary>힌트 보기</summary>

- Kafka 출력 시 필요한 컬럼: `key`, `value`
- `to_json(struct("*"))`: 모든 컬럼을 JSON 문자열로 변환
- `format("kafka")`와 `option("topic", "토픽명")` 사용

</details>

<details>
<summary>모범 답안</summary>

```python
from pyspark.sql.functions import to_json, struct

# =============================================================================
# Kafka Sink: 집계 결과를 다른 Kafka 토픽으로 전송
# =============================================================================

# 1. DataFrame을 Kafka 형식(key, value)으로 변환
df_output = (
    df_windowed
    .select(
        # key: 파티션 결정에 사용 (같은 key는 같은 파티션으로)
        col("endpoint").alias("key"),

        # value: 전체 데이터를 JSON 문자열로 변환
        # struct("*"): 모든 컬럼을 구조체로 묶음
        # to_json(): 구조체를 JSON 문자열로 변환
        to_json(struct("*")).alias("value")
    )
)

# 2. Kafka로 출력
query = (
    df_output.writeStream
    .outputMode("update")
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "api-events-aggregated")
    .option("checkpointLocation", "/data/checkpoints/kafka-sink")
    .trigger(processingTime="30 seconds")
    .start()
)

print("집계 결과를 Kafka로 전송 중!")
print("  소스 토픽: api-events")
print("  결과 토픽: api-events-aggregated")
query.awaitTermination()
```

**Kafka Consumer로 결과 확인**:

```bash
# 터미널에서 결과 토픽 구독
docker exec -it kafka kafka-console-consumer \
    --bootstrap-server localhost:9092 \
    --topic api-events-aggregated \
    --from-beginning
```

</details>

---

## 핵심 요약

### Structured Streaming 핵심 패턴

```python
# 1. Kafka에서 읽기
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "my-topic")
    .load()
)

# 2. 처리 (일반 DataFrame API 사용)
df_processed = df.select(...).filter(...).groupBy(...)

# 3. 출력
query = (
    df_processed.writeStream
    .outputMode("update")
    .format("console")  # 또는 "parquet", "kafka"
    .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()
```

### 윈도우 집계 패턴

```python
df.withWatermark("event_time", "10 minutes") \
  .groupBy(
      window(col("event_time"), "5 minutes"),  # 윈도우 크기
      col("group_column")
  ) \
  .agg(
      count("*").alias("count"),
      avg("value").alias("avg_value")
  )
```

### 출력 형식 비교

| 형식 | 용도 | 예시 |
|------|------|------|
| `console` | 개발/디버깅 | 실시간 로그 확인 |
| `parquet` | 데이터 저장 | 분석용 데이터 레이크 |
| `kafka` | 이벤트 전달 | 다운스트림 시스템 연동 |
| `memory` | 테스트 | 임시 테이블로 쿼리 |

### 버전 정보

| 컴포넌트 | 버전 |
|----------|------|
| Spark | 3.5.8 |
| spark-sql-kafka | 3.5.8 |
| Kafka | 4.1.1 |

---

## 다음 시간 예고

**6교시: 정리 및 보너스**

- 전체 아키텍처 리뷰
- 각 도구의 역할 정리
- 대안 도구 비교 (Polars, DuckDB, Redis Streams)
- 보너스 과제 안내

In [ ]:
# 세션 정리 (필요시)
# spark.stop()

# Day 12 - 5교시: 정리 및 보너스

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- 오늘 구축한 데이터 파이프라인의 전체 아키텍처를 설명할 수 있다
- Kafka, Spark의 역할과 선택 이유를 설명할 수 있다
- 대안 도구들(Polars, DuckDB, Redis Streams)의 트레이드오프를 이해할 수 있다
- 이후 커리큘럼(ELK, 데이터 웨어하우스)과의 연결점을 파악할 수 있다

---

## 사용 버전 정보

이 교안에서 사용한 도구들의 버전 정보입니다:

| 도구 | 버전 | 릴리스 | 비고 |
|------|------|--------|------|
| **Apache Kafka** | 4.1.1 | 2025년 11월 | KRaft 모드 기본 |
| **Apache Spark** | 3.5.8 | 2026년 1월 | 최신 안정 버전 |
| **PySpark** | 3.5.8 | 2026년 1월 | Spark와 버전 일치 |
| **spark-sql-kafka** | 3.5.8 | 2026년 1월 | Spark와 버전 일치 |
| **Python** | 3.12 | | slim 이미지 |
| **confluent-kafka** | latest | | Python Kafka 클라이언트 |
| **Kafka UI** | latest | | 웹 기반 모니터링 |

---

## Part 1: 전체 아키텍처 리뷰 (15분)

### 오늘 구축한 파이프라인

```
┌─────────────────────────────────────────────────────────────────────┐
│                        API Gateway 모니터링 시스템                    │
└─────────────────────────────────────────────────────────────────────┘

┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  API Events  │────→│    Kafka     │────→│    Spark     │
│  (Producer)  │     │              │     │  Streaming   │
└──────────────┘     └──────────────┘     └──────────────┘
                           │                     │
                           ↓                     ↓
                     ┌──────────┐         ┌──────────────┐
                     │ Kafka UI │         │  Parquet /   │
                     │ 모니터링   │         │  Kafka Sink  │
                     └──────────┘         └──────────────┘
```

### 각 컴포넌트의 역할

| 컴포넌트 | 역할 | 오늘 배운 것 |
|---------|------|-------------|
| **Producer** | 이벤트 생성 및 전송 | confluent-kafka, JSON 직렬화 |
| **Kafka** | 이벤트 버퍼링 및 분배 | KRaft 모드, 파티션, Consumer Group |
| **Spark** | 실시간/배치 데이터 처리 | DataFrame, Structured Streaming |
| **Kafka UI** | 클러스터 모니터링 | 토픽, 메시지, Consumer Lag |
| **Spark UI** | 작업 모니터링 | Job, Stage, Task |

### 데이터 흐름

```
1. 이벤트 발생
   API Gateway에서 요청/응답 정보 생성
   ↓
2. Kafka로 전송
   Producer가 JSON 직렬화 후 api-events 토픽에 전송
   파티션별로 분산 저장
   ↓
3. Spark에서 실시간 처리
   Structured Streaming으로 Kafka 구독
   윈도우 기반 집계 (5분/1분)
   ↓
4. 결과 출력
   콘솔: 실시간 모니터링
   Parquet: 분석용 저장
   Kafka: 다운스트림 시스템 연동
```

---

## Part 2: 도구별 역할 정리 (15분)

### Kafka: "왜 Kafka를 쓰나요?"

**문제 상황**:
```
[직접 연결 방식]
Producer → Consumer 직접 전송

문제점:
- Consumer가 죽으면? 데이터 유실
- Consumer가 느리면? Producer 대기
- 여러 Consumer가 필요하면? 복잡해짐
```

**Kafka 해결책**:
```
Producer → Kafka → Consumer

장점:
- Consumer가 죽어도 데이터 보존 (영속성)
- Producer/Consumer 독립적 확장 (디커플링)
- 여러 Consumer Group 지원 (팬아웃)
- 재처리 가능 (오프셋 관리)
```

### Spark: "왜 Spark를 쓰나요?"

**문제 상황**:
```
[Pandas 한계]
데이터 100만 건: 5초
데이터 1000만 건: 50초... 메모리 부족!
데이터 1억 건: Out of Memory
```

**Spark 해결책**:
```
[분산 처리]
데이터 100만 건: Worker 1대 → 3초
데이터 1000만 건: Worker 5대 → 6초
데이터 1억 건: Worker 50대 → 60초

스케일 아웃으로 처리량 선형 증가!
```

### 오늘 측정한 성능

| 지표 | 결과 | 의미 |
|------|------|------|
| Producer 처리량 | ~50,000 records/sec | 초당 5만 건 Kafka 전송 |
| Pandas vs Spark (100만건) | Spark가 1.5배 빠름 | 대용량에서 Spark 유리 |
| 파티션 1 vs 4 | 15% 처리량 향상 | 병렬화의 효과 |

---

## Part 3: 대안 도구 비교 (15분)

### "항상 Kafka + Spark가 정답인가요?"

아닙니다! 상황에 따라 더 적합한 도구가 있습니다.

### 대안 도구 비교표

| 도구 | vs | 장점 | 단점 | 언제 선택? |
|------|-----|------|------|-----------|
| **Polars** | Spark | 빠름, 설치 간단, API 직관적 | 분산 처리 X | 단일 머신으로 충분할 때 |
| **DuckDB** | Spark | SQL 친화적, 설치 불필요 | 분산 처리 X | 분석 쿼리 위주일 때 |
| **Redis Streams** | Kafka | 간단, 매우 빠름 | 영속성 제한, 기능 제한 | 간단한 실시간 처리 |

### Polars: "빠른 단일 머신 처리"

```python
import polars as pl

# Pandas보다 10~100배 빠름!
df = pl.read_csv("api_events_1m.csv")
result = df.group_by("endpoint").agg([
    pl.count("request_id").alias("count"),
    pl.mean("response_time_ms").alias("avg_time"),
])
```

**선택 기준**:
- 데이터가 단일 머신 메모리에 들어감 (< 100GB)
- 분산 환경 구축이 부담스러움
- 빠른 개발과 반복이 필요

### DuckDB: "SQL로 빠르게 분석"

```python
import duckdb

# SQL로 바로 분석!
result = duckdb.sql('''
    SELECT endpoint,
           COUNT(*) as count,
           AVG(response_time_ms) as avg_time
    FROM 'api_events_1m.csv'
    GROUP BY endpoint
''').df()
```

**선택 기준**:
- SQL에 익숙한 팀
- 분석 쿼리 위주 (OLAP)
- 설치/설정 최소화

### Redis Streams: "초경량 실시간 처리"

```python
import redis

r = redis.Redis()

# Producer
r.xadd("api-events", {"endpoint": "/api/users", "status": "200"})

# Consumer
messages = r.xread({"api-events": "0"}, count=100)
```

**선택 기준**:
- 아주 간단한 실시간 처리
- 이미 Redis 사용 중
- 영속성보다 속도가 중요

### 선택 가이드: 의사결정 트리

```
데이터 규모가 어느 정도인가?
│
├─ < 1GB → Pandas / Polars
│
├─ 1GB ~ 100GB
│   │
│   └─ 분산 환경 가능? ─┬─ Yes → Spark
│                       └─ No  → Polars / DuckDB
│
└─ > 100GB → Spark (분산 필수)

실시간 처리가 필요한가?
│
├─ Yes ─┬─ 복잡한 처리/조인 → Kafka + Spark Streaming
│       └─ 단순 처리      → Redis Streams
│
└─ No → 배치 처리 (Spark / Polars / DuckDB)
```

---

## Part 4: 이후 커리큘럼 연결 (10분)

### 다음에 배울 것들

| 주제 | 연결점 | 배우는 이유 |
|------|--------|------------|
| **ELK Stack** | Kafka → Logstash → Elasticsearch | 로그 검색/시각화 |
| **Data Warehouse** | Spark → Snowflake/BigQuery | 분석용 데이터 저장 |
| **Airflow** | Spark 작업 스케줄링 | 배치 파이프라인 자동화 |

### 오늘 배운 것의 확장

```
[오늘]
API Events → Kafka → Spark → 콘솔/Parquet

[확장 1: ELK]
API Events → Kafka → Logstash → Elasticsearch → Kibana
                                   (검색 가능)    (대시보드)

[확장 2: Data Warehouse]
API Events → Kafka → Spark → Snowflake → BI Tool
                              (SQL 분석)  (시각화)

[확장 3: 완전 자동화]
Airflow가 스케줄링
   ↓
Kafka → Spark (실시간)
Spark (배치) → Data Warehouse → 리포트 자동 생성
```

---

## Part 5: 보너스 과제 안내

| 과제 | 난이도 | 연습 개념 |
|------|--------|----------|
| 보너스 1 | ⭐⭐ | Kafka 클러스터, 복제, 고가용성 |
| 보너스 2 | ⭐ | 키 기반 파티셔닝, 메시지 순서 보장 |
| 보너스 3 | ⭐⭐ | Kafka Sink, to_json, struct |

---

### 보너스 1: 3대 브로커 클러스터 구성

**목표**: 고가용성을 위한 Kafka 클러스터 구성

**왜 필요한가?**:
- 단일 브로커 장애 시 서비스 중단 방지
- 데이터 복제로 유실 방지
- 프로덕션 환경의 기본 구성

<details>
<summary>힌트 보기</summary>

- `KAFKA_CONTROLLER_QUORUM_VOTERS`: 컨트롤러 투표자 목록
- `KAFKA_DEFAULT_REPLICATION_FACTOR`: 토픽 생성 시 기본 복제 수
- `KAFKA_MIN_INSYNC_REPLICAS`: 쓰기 성공에 필요한 최소 동기화 복제본 수

</details>

<details>
<summary>모범 답안</summary>

**파일**: `docker/multi-broker.yml`

```yaml
services:
  kafka-1:
    image: apache/kafka:4.1.1
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      # ... 나머지 설정

  kafka-2:
    image: apache/kafka:4.1.1
    environment:
      KAFKA_NODE_ID: 2
      # ... 동일 설정

  kafka-3:
    image: apache/kafka:4.1.1
    environment:
      KAFKA_NODE_ID: 3
      # ... 동일 설정
```

**실행**:
```bash
docker compose -f multi-broker.yml up -d
```

**확인 방법**:
```bash
# 브로커 목록 확인
docker exec -it kafka-1 kafka-broker-api-versions --bootstrap-server kafka-1:9092

# 토픽 복제 상태 확인
docker exec -it kafka-1 kafka-topics --describe --topic api-events --bootstrap-server kafka-1:9092
```

**주요 설정 설명**:

| 설정 | 값 | 의미 |
|------|-----|------|
| `REPLICATION_FACTOR: 3` | 3 | 각 파티션을 3개 브로커에 복제 |
| `MIN_INSYNC_REPLICAS: 2` | 2 | 최소 2개 복제본 동기화 필요 |

</details>

---

### 보너스 2: 키 기반 파티셔닝

**목표**: user_id를 키로 사용하여 같은 유저의 메시지가 같은 파티션으로 가도록 설정

**왜 필요한가?**:
- 같은 사용자의 이벤트 순서 보장
- 사용자별 세션 데이터 처리
- 상태 유지(stateful) 처리 최적화

<details>
<summary>힌트 보기</summary>

- `producer.produce()`의 `key` 파라미터 사용
- 키는 바이트로 인코딩 필요: `key.encode("utf-8")`
- Kafka는 키의 해시값으로 파티션 결정

</details>

<details>
<summary>모범 답안</summary>

```python
from confluent_kafka import Producer
import json
import random
from datetime import datetime

producer = Producer({"bootstrap.servers": "kafka:9092"})

# 같은 user_id는 같은 파티션으로
for i in range(100):
    event = {
        "request_id": f"REQ_{i:06d}",
        "user_id": f"U{random.randint(1, 10):04d}",  # 10명의 유저
        "endpoint": "/api/test",
        "timestamp": datetime.now().isoformat(),
    }

    # key 파라미터 추가!
    producer.produce(
        topic="api-events",
        key=event["user_id"].encode("utf-8"),  # 키로 파티션 결정
        value=json.dumps(event).encode("utf-8"),
    )

producer.flush()
print("키 기반 파티셔닝 완료!")
```

**확인 방법**:
```bash
# Kafka UI에서 확인하거나 CLI로 확인
docker exec -it kafka kafka-console-consumer \
    --bootstrap-server localhost:9092 \
    --topic api-events \
    --property print.key=true \
    --property print.partition=true \
    --from-beginning
```

**예상 결과**: 같은 user_id는 항상 같은 파티션 번호에 출력됨

</details>

---

### 보너스 3: Streaming 결과를 Kafka 토픽으로 전송

**목표**: 집계 결과를 `api-events-aggregated` 토픽으로 전송

**왜 필요한가?**:
- 다운스트림 시스템 연동 (알림, 대시보드 등)
- 마이크로서비스 간 이벤트 전달
- 데이터 파이프라인 확장

<details>
<summary>힌트 보기</summary>

- Kafka는 key-value 형태로 메시지 전송
- `to_json(struct("*"))`: DataFrame을 JSON 문자열로 변환
- `format("kafka")` + `option("topic", "토픽명")`
- 체크포인트 필수!

</details>

<details>
<summary>모범 답안</summary>

```python
from pyspark.sql.functions import col, to_json, struct

# 1. DataFrame을 Kafka 형식(key, value)으로 변환
df_output = (
    df_windowed
    .select(
        # key: 파티션 결정용 (선택사항)
        col("endpoint").alias("key"),

        # value: 전체 데이터를 JSON으로 변환
        to_json(struct("*")).alias("value")
    )
)

# 2. Kafka로 출력
query = (
    df_output.writeStream
    .outputMode("update")
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "api-events-aggregated")
    .option("checkpointLocation", "/data/checkpoints/kafka-sink")
    .trigger(processingTime="30 seconds")
    .start()
)

print("집계 결과를 Kafka로 전송 중!")
query.awaitTermination()
```

**확인 방법**:
```bash
# 결과 토픽 구독
docker exec -it kafka kafka-console-consumer \
    --bootstrap-server localhost:9092 \
    --topic api-events-aggregated \
    --from-beginning
```

**예상 출력** (JSON 형식):
```json
{"window":{"start":"2026-01-18T10:00:00","end":"2026-01-18T10:05:00"},"endpoint":"/api/users","request_count":142,"avg_response_time":267.1}
```

</details>

---

## 오늘 배운 것 총정리

### 핵심 명령어/코드

```bash
# Docker 환경
docker compose up -d
docker compose ps
docker compose logs -f kafka
docker compose exec python bash
```

```python
# Kafka Producer (confluent-kafka)
from confluent_kafka import Producer
producer = Producer({"bootstrap.servers": "kafka:9092"})
producer.produce(topic, value=json.dumps(data).encode())
producer.flush()

# Kafka Consumer (confluent-kafka)
from confluent_kafka import Consumer
consumer = Consumer({"bootstrap.servers": "kafka:9092", "group.id": "my-group"})
consumer.subscribe(["topic"])
msg = consumer.poll(1.0)

# Spark Batch (PySpark 3.5.8)
spark = SparkSession.builder.master("spark://spark-master:7077").getOrCreate()
df = spark.read.csv("file.csv", header=True)
df.groupBy("col").agg(count("*")).show()

# Spark Streaming (Kafka 연동 - spark-sql-kafka 3.5.8)
spark = SparkSession.builder \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8") \
    .getOrCreate()
df = spark.readStream.format("kafka").option("subscribe", "topic").load()
query = df.writeStream.format("console").start()
```

### 체크리스트

- [ ] Docker Compose로 Kafka + Spark 환경 구성
- [ ] Kafka Producer로 API 이벤트 전송
- [ ] Consumer로 메시지 수신 및 처리
- [ ] 파티션 수에 따른 처리량 변화 측정
- [ ] Pandas vs Spark 성능 비교
- [ ] Structured Streaming으로 실시간 처리
- [ ] 윈도우 집계 구현

---

## 마무리

### 오늘의 핵심 메시지

> "아, 이래서 이런 도구를 쓰는구나!"

- **Kafka**: 데이터 유실 방지, 시스템 간 디커플링
- **Spark**: 대용량 데이터 처리, 실시간 스트리밍
- **도구 선택**: 상황에 따라 적절한 도구를 선택하는 것이 중요

### 다음 시간

ELK Stack (Elasticsearch, Logstash, Kibana)
- 로그 수집 및 검색
- 대시보드 시각화
- Kafka와의 연동

수고하셨습니다!